In [ ]:
# ============================================================
# Imports
# ============================================================


# Single source of truth for reproducibility
RANDOM_SEED = 42

In [ ]:
from pathlib import Path

FIGURE_DIR = Path("./figures")
FIGURE_DIR.mkdir(exist_ok=True)


def save_fig(fig, name: str, formats=("pdf", "svg"), dpi: int = 300):
    """Save a matplotlib figure as vector format(s) — no pixelation on zoom/print."""
    for fmt in formats:
        path = FIGURE_DIR / f"{name}.{fmt}"
        fig.savefig(path, format=fmt, dpi=dpi, bbox_inches="tight", facecolor="white")
    print(f"✓ Saved {FIGURE_DIR}/{name}.{{{','.join(formats)}}}")

In [ ]:
# ============================================================
# Optimized PCA (From Scratch using NumPy)
# ============================================================

import numpy as np


class OptimizedPCA:
    def __init__(self, n_components=None):
        """
        Optimized PCA implementation from scratch using NumPy.

        Parameters
        ----------
        n_components : int | float | None
            int   -> keep exactly n components
            float -> retain variance fraction (0-1)
            None  -> keep all components
        """
        self.n_components = self._normalize_and_validate(n_components)

    @staticmethod
    def _normalize_and_validate(n_components):
        """Coerce numpy scalars to Python types and validate range/type."""
        if n_components is None:
            return None

        # Any numpy scalar (int64, float32, ...) exposes .item()
        if hasattr(n_components, "item"):
            n_components = n_components.item()

        if isinstance(n_components, bool):  # bool is a subclass of int
            raise TypeError(
                f"n_components must be int, float, or None; got {type(n_components)}"
            )

        if isinstance(n_components, int):
            if n_components <= 0:
                raise ValueError(f"n_components (int) must be > 0, got {n_components}")
            return n_components

        if isinstance(n_components, float):
            if not (0 < n_components <= 1):
                raise ValueError(
                    f"n_components (float) must be in (0, 1], got {n_components}"
                )
            return n_components

        raise TypeError(
            f"n_components must be int, float, or None; got {type(n_components)}"
        )

    # ---------------------------------------------------------
    # Fitting steps
    # ---------------------------------------------------------

    def _center_data(self, x):
        self.mean_ = np.mean(x, axis=0)
        return x - self.mean_

    def _compute_svd(self, x_centered):
        _, s, vt = np.linalg.svd(x_centered, full_matrices=False)
        return s, vt

    def _compute_variance(self, s, n_samples):
        var = (s**2) / (n_samples - 1)
        ratio = var / var.sum()
        return var, ratio

    def _select_components(self, ratio, n_features):
        if self.n_components is None:
            return n_features
        if isinstance(self.n_components, float):
            return int(np.argmax(np.cumsum(ratio) >= self.n_components)) + 1
        return int(self.n_components)

    def _apply_sign_flip(self, vt):
        idx = np.argmax(np.abs(vt), axis=1)
        signs = np.sign(vt[np.arange(vt.shape[0]), idx])
        return vt * signs[:, None]

    def fit(self, x):
        x = np.asarray(x, dtype=np.float64)
        n_samples, n_features = x.shape

        x_c = self._center_data(x)
        s, vt = self._compute_svd(x_c)
        var, ratio = self._compute_variance(s, n_samples)
        vt = self._apply_sign_flip(vt)

        k = self._select_components(ratio, n_features)
        self.components_ = vt[:k]
        self.explained_variance_ = var[:k]
        self.explained_variance_ratio_ = ratio[:k]
        self.n_components_ = k
        return self

    def transform(self, x):
        x = np.asarray(x, dtype=np.float64)
        return (x - self.mean_) @ self.components_.T

    def fit_transform(self, x):
        return self.fit(x).transform(x)

In [ ]:
# ============================================================
# Load and Prepare NSL-KDD Dataset
# ============================================================

import urllib.error
import urllib.request
import warnings
from io import StringIO
from pathlib import Path

import pandas as pd

# Column names
COLUMN_NAMES = [
    "duration",
    "protocol_type",
    "service",
    "flag",
    "src_bytes",
    "dst_bytes",
    "land",
    "wrong_fragment",
    "urgent",
    "hot",
    "num_failed_logins",
    "logged_in",
    "num_compromised",
    "root_shell",
    "su_attempted",
    "num_root",
    "num_file_creations",
    "num_shells",
    "num_access_files",
    "num_outbound_cmds",
    "is_host_login",
    "is_guest_login",
    "count",
    "srv_count",
    "serror_rate",
    "srv_serror_rate",
    "rerror_rate",
    "srv_rerror_rate",
    "same_srv_rate",
    "diff_srv_rate",
    "srv_diff_host_rate",
    "dst_host_count",
    "dst_host_srv_count",
    "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate",
    "dst_host_srv_serror_rate",
    "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
    "label",
    "difficulty_level",
]

# Configuration for dataset loading
DATASET_CONFIG = {
    "train_url": "https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTrain%2B.txt",
    "test_url": "https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTest%2B.txt",
    "timeout": 30,
    "cache_dir": Path("./data_cache"),
}

# Attack category mapping — comprehensive and type-safe
ATTACK_MAP = {
    "normal": "Normal",
    # DoS attacks (12 types)
    "neptune": "DoS",
    "back": "DoS",
    "land": "DoS",
    "pod": "DoS",
    "smurf": "DoS",
    "teardrop": "DoS",
    "mailbomb": "DoS",
    "apache2": "DoS",
    "processtable": "DoS",
    "udpstorm": "DoS",
    "worm": "DoS",
    # Probe attacks (6 types)
    "satan": "Probe",
    "ipsweep": "Probe",
    "nmap": "Probe",
    "portsweep": "Probe",
    "mscan": "Probe",
    "saint": "Probe",
    # R2L (Remote-to-Local) attacks (15 types)
    "guess_passwd": "R2L",
    "ftp_write": "R2L",
    "imap": "R2L",
    "phf": "R2L",
    "multihop": "R2L",
    "warezmaster": "R2L",
    "warezclient": "R2L",
    "spy": "R2L",
    "xlock": "R2L",
    "xsnoop": "R2L",
    "snmpguess": "R2L",
    "snmpgetattack": "R2L",
    "httptunnel": "R2L",
    "sendmail": "R2L",
    "named": "R2L",
    # U2R (User-to-Root) attacks (7 types)
    "buffer_overflow": "U2R",
    "loadmodule": "U2R",
    "rootkit": "U2R",
    "perl": "U2R",
    "sqlattack": "U2R",
    "xterm": "U2R",
    "ps": "U2R",
}

# Expected category counts for validation
EXPECTED_CATEGORIES = {"Normal", "DoS", "Probe", "R2L", "U2R"}


# ============================================================
# Helper Functions
# ============================================================


def _download_csv(url, timeout):
    """Download raw CSV text from a URL, raising RuntimeError on any failure."""
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            return response.read().decode("utf-8")
    except urllib.error.URLError as e:
        raise RuntimeError(f"Network error downloading from {url}: {e}") from e
    except Exception as e:
        raise RuntimeError(f"Failed to download from {url}: {e}") from e


def _load_from_cache(cache_path, verbose):
    """Return cached DataFrame if it exists and is readable, else None."""
    if not cache_path.exists():
        return None
    if verbose:
        print(f"✓ Loading from cache: {cache_path}")
    try:
        return pd.read_parquet(cache_path)
    except Exception as e:
        warnings.warn(f"Cache read failed ({e}), re-downloading...")
        return None


def _save_to_cache(df, cache_path, verbose):
    """Best-effort cache write; failures are warnings, not fatal errors."""
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(cache_path, compression="snappy", index=False)
        if verbose:
            print(f"✓ Cached to: {cache_path}")
    except Exception as e:
        warnings.warn(f"Failed to cache dataset to {cache_path}: {e}")


def _download_and_merge(config, verbose):
    """Download train + test splits and merge them into a single DataFrame."""
    if verbose:
        print("Downloading NSL-KDD dataset...")
    try:
        train_text = _download_csv(config["train_url"], config["timeout"])
        train_df = pd.read_csv(StringIO(train_text), names=COLUMN_NAMES)
        if verbose:
            print(f"  ✓ Train set: {len(train_df):,} samples")

        test_text = _download_csv(config["test_url"], config["timeout"])
        test_df = pd.read_csv(StringIO(test_text), names=COLUMN_NAMES)
        if verbose:
            print(f"  ✓ Test set: {len(test_df):,} samples")
    except RuntimeError:
        raise  # already a clean, descriptive error from _download_csv
    except pd.errors.ParserError as e:
        raise RuntimeError(f"Failed to parse NSL-KDD CSV (malformed data): {e}") from e
    except Exception as e:
        raise RuntimeError(
            f"Failed to process NSL-KDD dataset ({type(e).__name__}): {e}"
        ) from e

    return pd.concat([train_df, test_df], ignore_index=True)


def load_nsl_kdd(config=None, use_cache=False, verbose=True):
    """
    Download and merge NSL-KDD train & test datasets with caching support.

    Parameters
    ----------
    config : dict, optional
        Dataset configuration with 'train_url', 'test_url', 'timeout', 'cache_dir'.
        Defaults to DATASET_CONFIG.
    use_cache : bool, default=False
        If True, check cache_dir before downloading and save merged data.
    verbose : bool, default=True
        Print loading progress and cache status.

    Returns
    -------
    pd.DataFrame
        Merged train + test data (not label-processed).

    Raises
    ------
    RuntimeError
        If network request or file I/O fails.
    """
    config = config or DATASET_CONFIG
    cache_path = Path(config["cache_dir"]) / "nsl_kdd_merged.parquet"

    if use_cache:
        cached = _load_from_cache(cache_path, verbose)
        if cached is not None:
            return cached

    merged = _download_and_merge(config, verbose)

    if use_cache:
        _save_to_cache(merged, cache_path, verbose)

    return merged


def create_labels(df, attack_map=None):
    """
    Create binary and multiclass labels with validation and warning handling.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with 'label' column.
    attack_map : dict, optional
        Mapping from attack label strings to category names.
        Defaults to ATTACK_MAP.

    Returns
    -------
    pd.DataFrame
        Copy of input with added 'target' (binary) and 'category' (multiclass) columns.
    """
    attack_map = attack_map or ATTACK_MAP
    df = df.copy()

    # Binary target: 0=Normal, 1=Attack
    df["target"] = (df["label"] != "normal").astype(int)

    # Multiclass category
    df["category"] = df["label"].map(attack_map)

    # Warn about any label not present in attack_map
    unmapped_mask = df["category"].isna()
    if unmapped_mask.any():
        unmapped_counts = df.loc[unmapped_mask, "label"].value_counts()
        print(f"\n⚠️  WARNING: {len(unmapped_counts)} unmapped attack type(s) found:")
        for label, count in unmapped_counts.items():
            print(f"     '{label}': {count} samples")
        print("  → Assigning to 'Unknown' category\n")
        df["category"] = df["category"].fillna("Unknown")

    return df


def validate_dataset(df, categorical_cols=None, label_col="label"):
    """
    Validate dataset integrity and structure.

    Parameters
    ----------
    df : pd.DataFrame
        Dataframe to validate (should have 'target', 'category' columns).
    categorical_cols : list, optional
        Expected categorical columns to check for. Defaults to None (skip).
    label_col : str
        Name of the label column before mapping.

    Raises
    ------
    AssertionError
        If validation checks fail.
    """
    _validate_binary_target(df)
    _validate_category_column(df)
    _validate_no_nans(df, ["target", "category"])
    if categorical_cols:
        _validate_no_nans(df, categorical_cols, must_exist=True)

    print(f"✓ Dataset validation passed ({len(df):,} samples, {df.shape[1]} features)")


def _validate_binary_target(df):
    assert "target" in df.columns, "Missing 'target' column"
    assert (
        df["target"].isin([0, 1]).all()
    ), "Invalid binary target values (expected 0 or 1)"
    assert (df["target"] == 1).sum() > 0, "No attack samples (target=1) in dataset"
    assert (df["target"] == 0).sum() > 0, "No normal samples (target=0) in dataset"


def _validate_category_column(df):
    assert "category" in df.columns, "Missing 'category' column"
    unique_categories = set(df["category"].unique())
    invalid_categories = unique_categories - (EXPECTED_CATEGORIES | {"Unknown"})
    assert not invalid_categories, f"Unexpected categories found: {invalid_categories}"


def _validate_no_nans(df, columns, must_exist=False):
    for col in columns:
        if must_exist:
            assert col in df.columns, f"Missing categorical column '{col}'"
        assert not df[col].isna().any(), f"NaN values found in column '{col}'"


def print_dataset_summary(df, show_class_balance=True):
    """
    Print comprehensive dataset summary statistics.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset with 'target' and 'category' columns.
    show_class_balance : bool, default=True
        If True, show percentage distribution alongside counts.
    """
    print("\n" + "=" * 70)
    print("NSL-KDD DATASET SUMMARY")
    print("=" * 70)
    print(f"Total Samples : {len(df):,}")
    print(f"Total Features: {df.shape[1]}")
    print(f"Memory usage  : {df.memory_usage(deep=True).sum() / 1024 ** 2:.2f} MB")

    print("\n" + "-" * 70)
    print("BINARY CLASS DISTRIBUTION (Normal vs Attack)")
    print("-" * 70)
    binary_counts = df["target"].value_counts().sort_index()
    for class_val, label in {0: "Normal", 1: "Attack"}.items():
        count = binary_counts.get(class_val, 0)
        suffix = f" ({100 * count / len(df):6.2f}%)" if show_class_balance else ""
        print(f"  {label:10s}: {count:8,d}{suffix}")
    print(f"  Imbalance ratio: {binary_counts.max() / binary_counts.min():.2f}:1")

    print("\n" + "-" * 70)
    print("MULTICLASS DISTRIBUTION (5-Category Attack Types)")
    print("-" * 70)
    multiclass_counts = df["category"].value_counts()
    for category in sorted(multiclass_counts.index):
        count = multiclass_counts[category]
        if show_class_balance:
            pct = 100 * count / len(df)
            bar = "█" * int(pct / 2)  # scale to ~50 chars max
            print(f"  {category:10s}: {count:8,d} ({pct:6.2f}%) {bar}")
        else:
            print(f"  {category:10s}: {count:8,d}")

    print("=" * 70 + "\n")


# ============================================================
# Load and Prepare Dataset
# ============================================================


def load_and_prepare_dataset():
    """Run the full load -> label -> validate -> summarize pipeline."""
    full_df = load_nsl_kdd(config=DATASET_CONFIG, use_cache=False, verbose=True)
    full_df = create_labels(full_df, attack_map=ATTACK_MAP)
    full_df.drop(columns=["label", "difficulty_level"], inplace=True)

    validate_dataset(full_df)
    print_dataset_summary(full_df, show_class_balance=True)

    print(
        f"Dataset ready for preprocessing.\n"
        f"Shape: {full_df.shape}\n"
        f"Dtypes:\n{full_df.dtypes}\n"
    )
    return full_df


full_df = load_and_prepare_dataset()

In [ ]:
# ============================================================
# Encode Categorical Features
# ============================================================

import json
import warnings
from pathlib import Path
from typing import Dict

import joblib
import pandas as pd
from sklearn.preprocessing import LabelEncoder

categorical_columns = [
    "protocol_type",
    "service",
    "flag",
]

ENCODER_SAVE_PATH = Path("./encoders")

# ============================================================
# EncoderManager Class — Reproducible Encoding & Inference
# ============================================================


class EncoderManager:
    """
    Manages categorical feature encoding with save/load support.

    Enables reproducible preprocessing for:
    - Training data (fit_transform)
    - Validation/test data (transform)
    - Production inference (load + transform)

    Handles unseen categories gracefully with configurable fallback.
    """

    def __init__(self, categorical_features: list):
        """
        Initialize encoder manager.

        Parameters
        ----------
        categorical_features : list
            Column names to encode (e.g., ['protocol_type', 'service', 'flag'])
        """
        self.categorical_features = categorical_features
        self.encoders: Dict[str, LabelEncoder] = {
            col: LabelEncoder() for col in categorical_features
        }
        self.is_fitted = False

    def _validate_columns(self, df: pd.DataFrame) -> None:
        """Ensure required columns exist and contain no NaNs. Shared by fit/transform."""
        missing_cols = set(self.categorical_features) - set(df.columns)
        if missing_cols:
            raise ValueError(f"Missing columns in dataframe: {missing_cols}")

        for col in self.categorical_features:
            if df[col].isna().any():
                raise ValueError(
                    f"Column '{col}' contains NaN values. "
                    "Handle missing values before encoding."
                )

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Fit encoders on data and transform in one step.

        Parameters
        ----------
        df : pd.DataFrame
            Input dataframe (must contain all categorical_features columns)

        Returns
        -------
        pd.DataFrame
            Copy of input with encoded categorical columns

        Raises
        ------
        ValueError
            If any categorical column is missing or contains NaN
        """
        self._validate_columns(df)
        df = df.copy()

        for col in self.categorical_features:
            df[col] = self.encoders[col].fit_transform(df[col])

        self.is_fitted = True
        return df

    def _handle_unseen_categories(
        self, df: pd.DataFrame, col: str, handle_unseen: str
    ) -> None:
        """Detect, report, and placeholder-fill unseen categories for a single column in-place."""
        known_classes = set(self.encoders[col].classes_)
        unseen_mask = ~df[col].isin(known_classes)
        if not unseen_mask.any():
            return

        unseen_values = df.loc[unseen_mask, col].unique()
        msg = (
            f"⚠️  Column '{col}': {unseen_mask.sum()} rows with unseen "
            f"categories {set(unseen_values)}"
        )
        if handle_unseen == "error":
            raise ValueError(msg)
        if handle_unseen == "warn":
            warnings.warn(msg)
        # 'ignore' -> silently continue

        # Replace unseen values with the first known class as a fallback before encoding
        fallback_val = self.encoders[col].classes_[0]
        df.loc[unseen_mask, col] = fallback_val

    def transform(self, df: pd.DataFrame, handle_unseen: str = "warn") -> pd.DataFrame:
        """
        Apply fitted encoders to new data.

        Parameters
        ----------
        df : pd.DataFrame
            Input dataframe (must contain all categorical_features columns)
        handle_unseen : str, default='warn'
            How to handle unseen categories:
            - 'warn': Print warning and use -1 as placeholder
            - 'error': Raise ValueError
            - 'ignore': Silently use -1 (for production)

        Returns
        -------
        pd.DataFrame
            Copy of input with encoded categorical columns

        Raises
        ------
        RuntimeError
            If encoders not fitted yet
        ValueError
            If handle_unseen='error' and unseen categories encountered
        """
        if not self.is_fitted:
            raise RuntimeError("Encoders not fitted yet. Call fit_transform() first.")

        self._validate_columns(df)
        df = df.copy()

        for col in self.categorical_features:
            self._handle_unseen_categories(df, col, handle_unseen)
            df[col] = self.encoders[col].transform(df[col])

        return df

    def get_mapping(self, column: str) -> Dict:
        """
        Get the encode-to-integer mapping for a column.

        Parameters
        ----------
        column : str
            Categorical column name

        Returns
        -------
        dict
            Mapping from original category string to encoded integer

        Example
        -------
        >>> manager.get_mapping('protocol_type')
        {'tcp': 0, 'udp': 1, 'icmp': 2}
        """
        if not self.is_fitted:
            raise RuntimeError("Encoders not fitted yet")
        if column not in self.encoders:
            raise ValueError(f"Column '{column}' not managed by this encoder")

        encoder = self.encoders[column]
        return dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))

    def save(self, path: Path = None) -> Path:
        """
        Persist encoders to disk for production use.

        Parameters
        ----------
        path : Path or str, optional
            Directory to save encoders. Defaults to ENCODER_SAVE_PATH.

        Returns
        -------
        Path
            Path where encoders were saved
        """
        if not self.is_fitted:
            raise RuntimeError("Cannot save unfitted encoders")

        path = Path(path) if path is not None else ENCODER_SAVE_PATH
        path.mkdir(parents=True, exist_ok=True)

        for col, encoder in self.encoders.items():
            joblib.dump(encoder, path / f"{col}.pkl")

        metadata = {
            "categorical_features": self.categorical_features,
            "encoders_saved_at": pd.Timestamp.now().isoformat(),
        }
        with open(path / "metadata.json", "w") as f:
            json.dump(metadata, f, indent=2)

        print(f"✓ Encoders saved to: {path}")
        return path

    @staticmethod
    def load(path: Path = None) -> "EncoderManager":
        """
        Load encoders from disk.

        Parameters
        ----------
        path : Path or str, optional
            Directory containing saved encoders. Defaults to ENCODER_SAVE_PATH.

        Returns
        -------
        EncoderManager
            Fitted encoder manager instance

        Raises
        ------
        FileNotFoundError
            If encoder files not found at path
        """
        path = Path(path) if path is not None else ENCODER_SAVE_PATH

        metadata_path = path / "metadata.json"
        if not metadata_path.exists():
            raise FileNotFoundError(
                f"Metadata not found at {metadata_path}. Invalid encoder directory."
            )

        with open(metadata_path, "r") as f:
            metadata = json.load(f)

        categorical_features = metadata["categorical_features"]
        manager = EncoderManager(categorical_features)

        for col in categorical_features:
            encoder_path = path / f"{col}.pkl"
            if not encoder_path.exists():
                raise FileNotFoundError(
                    f"Encoder for '{col}' not found at {encoder_path}"
                )
            manager.encoders[col] = joblib.load(encoder_path)

        manager.is_fitted = True
        print(f"✓ Encoders loaded from: {path}")
        return manager

    def __repr__(self) -> str:
        """String representation showing fitted status and managed columns."""
        status = "fitted" if self.is_fitted else "unfitted"
        return f"EncoderManager({status}, features={self.categorical_features})"


# ============================================================
# Reporting helpers (script section)
# ============================================================


def _print_encoding_mappings(encoder_manager: "EncoderManager", columns: list) -> None:
    print("\n" + "-" * 70)
    print("ENCODING MAPPINGS")
    print("-" * 70)
    for col in columns:
        mapping = encoder_manager.get_mapping(col)
        print(f"\n{col}:")
        for original, encoded in sorted(mapping.items(), key=lambda x: x[1]):
            print(f"  {original:20s} → {encoded}")


def _print_class_mapping(class_names, y_multiclass) -> None:
    print("\nClass Encoding Mapping:")
    for encoded_val, class_label in enumerate(class_names):
        count = (y_multiclass == encoded_val).sum()
        pct = 100 * count / len(y_multiclass)
        print(f"  {encoded_val}: {class_label:10s} (n={count:6,d}, {pct:6.2f}%)")
    print(f"\nClass Names Array: {class_names}")


def _print_feature_matrix_summary(x_all, y_binary, y_multiclass, class_names) -> None:
    print(f"\nFeature Matrix Dimensions")
    print("-" * 70)
    print(f"Samples           : {x_all.shape[0]:,}")
    print(f"Features          : {x_all.shape[1]}")
    print(f"Feature dtype     : {x_all.dtypes.unique()}")
    print(f"Missing values    : {x_all.isna().sum().sum()}")

    print(f"\nLabel Vectors")
    print("-" * 70)
    print(
        f"Binary labels     : {y_binary.shape[0]:,} samples (dtype: {y_binary.dtype})"
    )
    print(f"  • Class 0 (Normal) : {(y_binary == 0).sum():,}")
    print(f"  • Class 1 (Attack) : {(y_binary == 1).sum():,}")
    print(
        f"Multiclass labels : {y_multiclass.shape[0]:,} samples (dtype: {y_multiclass.dtype})"
    )
    for i, cls in enumerate(class_names):
        print(f"  • Class {i} ({cls:10s}): {(y_multiclass == i).sum():,}")

    print(f"\nFeature Names ({x_all.shape[1]} total)")
    print("-" * 70)
    feature_names = list(x_all.columns)
    for i in range(0, len(feature_names), 3):
        batch = feature_names[i : i + 3]
        print("  " + " | ".join(f"{f:20s}" for f in batch))


# ============================================================
# Initialize and Fit Feature Encoders
# ============================================================


def encode_features_and_labels(full_df: pd.DataFrame):
    """
    Run the full categorical-encoding pipeline: fit feature encoders,
    encode the multiclass label, and assemble the final feature matrix.

    Returns
    -------
    x_all, y_all_binary, y_all_multiclass, class_names, encoder_manager
    """
    print("=" * 70)
    print("CATEGORICAL FEATURE ENCODING")
    print("=" * 70)

    encoder_manager = EncoderManager(categorical_columns)
    full_df_encoded = encoder_manager.fit_transform(full_df)

    _print_encoding_mappings(encoder_manager, categorical_columns)
    encoder_manager.save(ENCODER_SAVE_PATH)
    print("\n" + "=" * 70)

    print("\nMULTICLASS LABEL ENCODING")
    print("-" * 70)
    multiclass_encoder = LabelEncoder()
    y_all_multiclass = multiclass_encoder.fit_transform(full_df_encoded["category"])
    class_names = multiclass_encoder.classes_
    _print_class_mapping(class_names, y_all_multiclass)

    print("\n" + "=" * 70)
    print("FEATURE MATRIX PREPARATION")
    print("=" * 70)

    # "category" excluded to prevent label leakage; use the encoded dataframe
    x_all = full_df_encoded.drop(columns=["target", "category"])
    y_all_binary = full_df_encoded["target"]

    assert x_all.shape[0] == y_all_binary.shape[0], "Feature matrix and label mismatch"
    assert (
        x_all.shape[0] == y_all_multiclass.shape[0]
    ), "Feature matrix and multiclass label mismatch"

    _print_feature_matrix_summary(x_all, y_all_binary, y_all_multiclass, class_names)

    print("\n" + "=" * 70)
    print("✓ Feature matrix ready for scaling and model training")
    print("=" * 70 + "\n")

    return x_all, y_all_binary, y_all_multiclass, class_names, encoder_manager


x_all, y_all_binary, y_all_multiclass, class_names, encoder_manager = (
    encode_features_and_labels(full_df)
)

In [ ]:
# ============================================================
# Train / Validation / Test Split with Stratification
# ============================================================

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def _render_bar(pct: float, scale: int = 2) -> str:
    """Build a simple text progress bar for a percentage (0-100)."""
    return "█" * int(pct / scale)


class DataSplitter:
    """
    Manages stratified train/validation/test splits with reproducibility.

    Ensures:
    - Stratified splits preserve class distribution across all sets
    - No data leakage (indices never mix)
    - Reproducible with RANDOM_SEED
    - Clear logging of split statistics
    """

    def __init__(self, random_state=RANDOM_SEED):
        """Initialize with fixed random seed for reproducibility."""
        self.random_state = random_state
        self.train_idx = None
        self.val_idx = None
        self.test_idx = None
        self.split_stats = {}

    def split(
        self,
        x_data: pd.DataFrame,
        y_data: pd.Series,
        train_size: float = 0.60,
        val_size: float = 0.20,
        test_size: float = 0.20,
        stratify_by: pd.Series = None,
    ) -> tuple:
        stratify_by = y_data if stratify_by is None else stratify_by
        self._validate_split_inputs(
            x_data, y_data, stratify_by, train_size, val_size, test_size
        )

        # Step 1: split test from train+val — split x, y, AND stratify_by together
        # so all three remain positionally aligned.
        x_train_val, x_test, y_train_val, y_test, stratify_train_val, _ = (
            train_test_split(
                x_data,
                y_data,
                stratify_by,
                test_size=test_size,
                stratify=stratify_by,
                random_state=self.random_state,
            )
        )

        # Step 2: split train from val (val_fraction of the remaining train+val)
        val_fraction = val_size / (train_size + val_size)
        x_train, x_val, y_train, y_val = train_test_split(
            x_train_val,
            y_train_val,
            test_size=val_fraction,
            stratify=stratify_train_val,
            random_state=self.random_state,
        )

        self.train_idx = x_train.index.values
        self.val_idx = x_val.index.values
        self.test_idx = x_test.index.values

        self._compute_split_stats(y_train, y_val, y_test, stratify_by)

        return x_train, x_val, x_test, y_train, y_val, y_test

    @staticmethod
    def _validate_split_inputs(
        x_data, y_data, stratify_by, train_size, val_size, test_size
    ):
        total_size = train_size + val_size + test_size
        assert np.isclose(
            total_size, 1.0, atol=0.01
        ), f"Split sizes must sum to 1.0, got {total_size}"
        assert len(x_data) == len(y_data), "X and y must have same length"
        assert len(x_data) == len(
            stratify_by
        ), "x_data and stratify_by must have same length"

    def _compute_split_stats(self, y_train, y_val, y_test, stratify_by):
        """Compute distribution statistics for logging."""
        self.split_stats = {
            "train": self._class_distribution(y_train),
            "val": self._class_distribution(y_val),
            "test": self._class_distribution(y_test),
            "stratify_by": self._class_distribution(stratify_by),
        }

    @staticmethod
    def _class_distribution(y):
        """Get count and percentage for each class."""
        counts = y.value_counts().sort_index()
        total = len(y)
        return {
            cls: {"count": int(count), "pct": 100 * count / total}
            for cls, count in counts.items()
        }

    def print_split_summary(self):
        """Print detailed split statistics with class distribution."""
        total_samples = sum(
            d["count"]
            for split_name in ["train", "val", "test"]
            for d in self.split_stats[split_name].values()
        )

        print("\n" + "=" * 80)
        print("TRAIN / VALIDATION / TEST SPLIT SUMMARY")
        print("=" * 80)

        print(f"\nOverall Split Ratio")
        print("-" * 80)
        for split_name in ["train", "val", "test"]:
            split_total = sum(d["count"] for d in self.split_stats[split_name].values())
            pct_of_total = 100 * split_total / total_samples
            print(
                f"  {split_name.upper():10s}: {split_total:8,d} samples ({pct_of_total:6.2f}%)"
            )

        print(f"\nClass Distribution by Split (Binary: 0=Normal, 1=Attack)")
        print("-" * 80)
        for split_name in ["train", "val", "test"]:
            print(f"\n  {split_name.upper()}:")
            for cls, stats in self.split_stats[split_name].items():
                class_label = "Normal" if cls == 0 else "Attack"
                bar = _render_bar(stats["pct"])
                print(
                    f"    Class {cls} ({class_label:6s}): {stats['count']:8,d} ({stats['pct']:6.2f}%) {bar}"
                )

            counts = [stats["count"] for stats in self.split_stats[split_name].values()]
            imbalance_ratio = max(counts) / min(counts) if len(counts) > 1 else 1.0
            print(f"    Imbalance ratio: {imbalance_ratio:.2f}:1")

        print("\n" + "=" * 80 + "\n")


class ScalerManager:
    """
    Manages feature scaling with save/load for production.

    Ensures:
    - Scaler fit ONLY on training data (no data leakage)
    - Same scaling applied to val/test
    - Saved for consistent preprocessing in production
    """

    _SCALER_FACTORIES = (
        {}
    )  # populated lazily below to avoid importing every scaler upfront

    def __init__(self, scaler_type: str = "standard"):
        """
        Initialize scaler manager.

        Parameters
        ----------
        scaler_type : str, default='standard'
            Type of scaler: 'standard' (StandardScaler), 'minmax' (MinMaxScaler),
            'robust' (RobustScaler)
        """
        self.scaler_type = scaler_type
        self.scaler = self._build_scaler(scaler_type)
        self.is_fitted = False
        self.scaling_stats = {}

    @staticmethod
    def _build_scaler(scaler_type: str):
        if scaler_type == "standard":
            return StandardScaler()
        if scaler_type == "minmax":
            from sklearn.preprocessing import MinMaxScaler

            return MinMaxScaler()
        if scaler_type == "robust":
            from sklearn.preprocessing import RobustScaler

            return RobustScaler()
        raise ValueError(f"Unknown scaler_type: {scaler_type}")

    @staticmethod
    def _validate_no_nan_inf(x: np.ndarray, context: str) -> None:
        """Shared guard used by both fit_transform and transform."""
        if np.isnan(x).any():
            raise ValueError(f"{context} contains NaN values")
        if np.isinf(x).any():
            raise ValueError(f"{context} contains infinite values")

    def fit_transform(self, x_train: pd.DataFrame) -> np.ndarray:
        """
        Fit scaler on training data and transform in one step.

        Parameters
        ----------
        x_train : pd.DataFrame or np.ndarray
            Training features (will fit scaler on these)

        Returns
        -------
        np.ndarray
            Scaled training data

        Raises
        ------
        ValueError
            If data contains NaN or infinite values
        """
        x_train = np.asarray(x_train, dtype=np.float64)
        self._validate_no_nan_inf(x_train, "Training data")

        x_train_scaled = self.scaler.fit_transform(x_train)
        self.is_fitted = True
        self._compute_scaling_stats(x_train, x_train_scaled)

        return x_train_scaled

    def transform(self, x_data: pd.DataFrame) -> np.ndarray:
        """
        Apply fitted scaler to new data.

        Parameters
        ----------
        x_data : pd.DataFrame or np.ndarray
            Features to scale (must have same structure as training data)

        Returns
        -------
        np.ndarray
            Scaled data

        Raises
        ------
        RuntimeError
            If scaler not fitted yet
        ValueError
            If data contains NaN or infinite values
        """
        if not self.is_fitted:
            raise RuntimeError(
                "Scaler not fitted yet. Call fit_transform() on training data first."
            )

        x_data = np.asarray(x_data, dtype=np.float64)
        self._validate_no_nan_inf(x_data, "Data to transform")

        return self.scaler.transform(x_data)

    def _compute_scaling_stats(self, x_original, x_scaled):
        """Compute pre/post scaling statistics."""
        self.scaling_stats = {
            "original": self._describe(x_original),
            "scaled": self._describe(x_scaled),
        }

    @staticmethod
    def _describe(x: np.ndarray) -> dict:
        return {
            "mean": np.mean(x, axis=0),
            "std": np.std(x, axis=0),
            "min": np.min(x, axis=0),
            "max": np.max(x, axis=0),
        }

    def print_scaling_summary(self, feature_names=None, top_n=10):
        """
        Print scaling statistics (before/after).

        Parameters
        ----------
        feature_names : list, optional
            Feature names for readability
        top_n : int, default=10
            Show top N features by original variance
        """
        if not self.is_fitted:
            print("Scaler not fitted yet")
            return

        print("\n" + "=" * 80)
        print(f"SCALING SUMMARY ({self.scaler_type.upper()})")
        print("=" * 80)

        original = self.scaling_stats["original"]
        scaled = self.scaling_stats["scaled"]

        variances = original["std"] ** 2
        top_indices = np.argsort(variances)[-top_n:][::-1]

        print(f"\nTop {top_n} Features by Original Variance")
        print("-" * 80)
        print(
            f"{'Feature':<25} | {'Original Mean':>12} | {'Original Std':>12} | "
            f"{'Scaled Mean':>12} | {'Scaled Std':>12}"
        )
        print("-" * 80)

        for idx in top_indices:
            feat_name = feature_names[idx] if feature_names else f"Feature_{idx}"
            print(
                f"{feat_name:<25} | {original['mean'][idx]:12.4f} | "
                f"{original['std'][idx]:12.4f} | {scaled['mean'][idx]:12.4f} | "
                f"{scaled['std'][idx]:12.4f}"
            )

        print("\n" + "=" * 80 + "\n")

    def save(self, path: Path = None):
        """Save fitted scaler to disk."""
        if not self.is_fitted:
            raise RuntimeError("Cannot save unfitted scaler")

        path = Path(path) if path is not None else Path("./scalers")
        path.mkdir(parents=True, exist_ok=True)

        scaler_path = path / f"scaler_{self.scaler_type}.pkl"
        joblib.dump(self.scaler, scaler_path)

        print(f"✓ Scaler saved to: {scaler_path}")
        return scaler_path

    @staticmethod
    def load(path: Path = None, scaler_type: str = "standard"):
        """Load fitted scaler from disk."""
        path = Path(path) if path is not None else Path("./scalers")
        scaler_path = path / f"scaler_{scaler_type}.pkl"

        if not scaler_path.exists():
            raise FileNotFoundError(f"Scaler not found at {scaler_path}")

        manager = ScalerManager(scaler_type)
        manager.scaler = joblib.load(scaler_path)
        manager.is_fitted = True

        print(f"✓ Scaler loaded from: {scaler_path}")
        return manager

    def __repr__(self) -> str:
        status = "fitted" if self.is_fitted else "unfitted"
        return f"ScalerManager({self.scaler_type}, {status})"


# ============================================================
# Reporting helpers (script section)
# ============================================================


def _print_scaled_data_validation(
    x_tr_scaled, x_val_scaled, x_test_scaled, y_tr, y_val, y_test
):
    print("=" * 80)
    print("SCALED DATA VALIDATION")
    print("=" * 80)

    print(f"\nScaled Data Shapes")
    print("-" * 80)
    print(f"  x_tr_scaled : {x_tr_scaled.shape}")
    print(f"  x_val_scaled: {x_val_scaled.shape}")
    print(f"  x_test_scaled: {x_test_scaled.shape}")

    print(f"\nScaled Data Statistics (Training Set)")
    print("-" * 80)
    print(f"  Mean ≈ 0: {np.allclose(np.mean(x_tr_scaled, axis=0), 0, atol=1e-10)}")
    print(f"  Std ≈ 1:  {np.allclose(np.std(x_tr_scaled, axis=0), 1, atol=0.01)}")
    print(f"  Min: {np.min(x_tr_scaled):.4f}")
    print(f"  Max: {np.max(x_tr_scaled):.4f}")
    print(f"  NaN count: {np.isnan(x_tr_scaled).sum()}")

    print(f"\nLabel Distribution (Scaled Data)")
    print("-" * 80)
    for set_name, y in [("Train", y_tr), ("Val", y_val), ("Test", y_test)]:
        c0, c1 = (y == 0).sum(), (y == 1).sum()
        total = len(y)
        print(
            f"  {set_name:5s}: Class 0={c0:6,d} ({100*c0/total:5.2f}%) | "
            f"Class 1={c1:6,d} ({100*c1/total:5.2f}%)"
        )

    print("\n" + "=" * 80)
    print("✓ Data splitting and scaling complete")
    print("=" * 80 + "\n")


# ============================================================
# Execute Split + Scaling Pipeline
# ============================================================


def split_and_scale_data(
    x_all: pd.DataFrame, y_all_binary: pd.Series, random_state=RANDOM_SEED
):
    """
    Run stratified train/val/test splitting followed by feature scaling.

    Returns
    -------
    dict with keys: x_tr_scaled, x_val_scaled, x_test_scaled, y_tr, y_val, y_test,
    scaler_manager, splitter
    """
    print("=" * 80)
    print("STRATIFIED DATA SPLITTING")
    print("=" * 80)

    splitter = DataSplitter(random_state=random_state)
    x_tr, x_val, x_test, y_tr, y_val, y_test = splitter.split(
        x_data=x_all,
        y_data=y_all_binary,
        train_size=0.60,
        val_size=0.20,
        test_size=0.20,
        stratify_by=y_all_binary,
    )
    splitter.print_split_summary()

    print("=" * 80)
    print("FEATURE SCALING (StandardScaler)")
    print("=" * 80)
    print(
        "\n⚠️  Scaler fit ONLY on training data to prevent data leakage"
        "\n    Same transform applied to validation & test sets\n"
    )

    scaler_manager = ScalerManager(scaler_type="standard")
    x_tr_scaled = scaler_manager.fit_transform(x_tr.astype(np.float64))
    x_val_scaled = scaler_manager.transform(x_val.astype(np.float64))
    x_test_scaled = scaler_manager.transform(x_test.astype(np.float64))

    scaler_manager.print_scaling_summary(feature_names=list(x_all.columns), top_n=15)
    scaler_manager.save(Path("./scalers"))

    _print_scaled_data_validation(
        x_tr_scaled, x_val_scaled, x_test_scaled, y_tr, y_val, y_test
    )

    return {
        "x_tr_scaled": x_tr_scaled,
        "x_val_scaled": x_val_scaled,
        "x_test_scaled": x_test_scaled,
        "y_tr": y_tr,
        "y_val": y_val,
        "y_test": y_test,
        "scaler_manager": scaler_manager,
        "splitter": splitter,
    }


split_result = split_and_scale_data(x_all, y_all_binary, random_state=RANDOM_SEED)
x_tr_scaled = split_result["x_tr_scaled"]
x_val_scaled = split_result["x_val_scaled"]
x_test_scaled = split_result["x_test_scaled"]
y_tr = split_result["y_tr"]
y_val = split_result["y_val"]
y_test = split_result["y_test"]
scaler_manager = split_result["scaler_manager"]
splitter = split_result["splitter"]

print("Available variables for downstream cells:")
for key in split_result:
    print(f"  • {key}")

In [ ]:
# ============================================================
# Principal Component Analysis (PCA) — Enhanced with Diagnostics
# ============================================================

import time
from typing import Tuple, Dict


class PCAComparator:
    """
    Compares sklearn PCA with custom NumPy implementation.

    Validates:
    - Numerical equivalence (within floating-point precision)
    - Performance characteristics
    - Variance retention
    - Component stability across implementations
    """

    def __init__(self, random_state=RANDOM_SEED):
        """Initialize PCA comparator."""
        self.random_state = random_state
        self.sklearn_pca = None
        self.custom_pca = None
        self.comparison_results = {}

    def fit_and_compare(
        self,
        x_train_scaled: np.ndarray,
        x_val_scaled: np.ndarray,
        x_test_scaled: np.ndarray,
        n_components: str = "mle",
        verbose: bool = True,
    ) -> Dict:
        """
        Fit both PCA implementations and compare results.

        Parameters
        ----------
        x_train_scaled : np.ndarray
            Scaled training features
        x_val_scaled : np.ndarray
            Scaled validation features
        x_test_scaled : np.ndarray
            Scaled test features
        n_components : str or int, default='mle'
            Number of components ('mle' uses Minka's MLE heuristic)
        verbose : bool, default=True
            Print detailed comparison results

        Returns
        -------
        dict
            Comparison statistics including equivalence metrics
        """
        if verbose:
            print("=" * 80)
            print("PRINCIPAL COMPONENT ANALYSIS (PCA) COMPARISON")
            print("=" * 80)

        # ====== Fit Scikit-learn PCA ======
        if verbose:
            print("\n(A) SCIKIT-LEARN PCA")
            print("-" * 80)

        start_time = time.time()
        self.sklearn_pca = SklearnPCA(
            n_components=n_components,
            svd_solver="full",
            random_state=self.random_state,
        )
        x_tr_pca_sklearn = self.sklearn_pca.fit_transform(x_train_scaled)
        x_val_pca_sklearn = self.sklearn_pca.transform(x_val_scaled)
        x_test_pca_sklearn = self.sklearn_pca.transform(x_test_scaled)
        sklearn_runtime = time.time() - start_time

        sklearn_n_components = self.sklearn_pca.n_components_
        sklearn_explained_var = self.sklearn_pca.explained_variance_ratio_.sum()

        if verbose:
            print(f"Components selected    : {sklearn_n_components}")
            print(
                f"Total variance retained: {sklearn_explained_var:.4f} ({sklearn_explained_var*100:.2f}%)"
            )
            print(f"Execution time         : {sklearn_runtime:.4f} seconds")

        # ====== Fit Custom PCA ======
        if verbose:
            print("\n(B) CUSTOM PCA (NumPy IMPLEMENTATION)")
            print("-" * 80)

        start_time = time.time()
        self.custom_pca = OptimizedPCA(n_components=sklearn_n_components)
        x_tr_pca_custom = self.custom_pca.fit_transform(x_train_scaled)
        x_val_pca_custom = self.custom_pca.transform(x_val_scaled)
        x_test_pca_custom = self.custom_pca.transform(x_test_scaled)
        custom_runtime = time.time() - start_time

        custom_n_components = self.custom_pca.n_components_
        custom_explained_var = self.custom_pca.explained_variance_ratio_.sum()

        if verbose:
            print(f"Components selected    : {custom_n_components}")
            print(
                f"Total variance retained: {custom_explained_var:.4f} ({custom_explained_var*100:.2f}%)"
            )
            print(f"Execution time         : {custom_runtime:.4f} seconds")

        # ====== Numerical Verification ======
        if verbose:
            print("\n(C) NUMERICAL EQUIVALENCE VERIFICATION")
            print("-" * 80)

        # Compare transformed data (use absolute value to account for sign flips)
        diff_train = np.abs(np.abs(x_tr_pca_sklearn) - np.abs(x_tr_pca_custom))
        diff_val = np.abs(np.abs(x_val_pca_sklearn) - np.abs(x_val_pca_custom))
        diff_test = np.abs(np.abs(x_test_pca_sklearn) - np.abs(x_test_pca_custom))

        max_diff_train = np.max(diff_train)
        max_diff_val = np.max(diff_val)
        max_diff_test = np.max(diff_test)
        max_diff_overall = max(max_diff_train, max_diff_val, max_diff_test)

        mean_diff_train = np.mean(diff_train)
        mean_diff_val = np.mean(diff_val)
        mean_diff_test = np.mean(diff_test)

        # Compare explained variance
        var_diff = np.abs(sklearn_explained_var - custom_explained_var)

        if verbose:
            print("\nTransformed Data Comparison (max absolute differences)")
            print(f"  Train set: {max_diff_train:.2e} (mean: {mean_diff_train:.2e})")
            print(f"  Val set:   {max_diff_val:.2e} (mean: {mean_diff_val:.2e})")
            print(f"  Test set:  {max_diff_test:.2e} (mean: {mean_diff_test:.2e})")
            print(f"  Overall max: {max_diff_overall:.2e}")

            print("\nExplained Variance Comparison")
            print(f"  sklearn: {sklearn_explained_var:.8f}")
            print(f"  custom:  {custom_explained_var:.8f}")
            print(f"  Diff:    {var_diff:.2e}")

            print("\nPerformance Comparison")
            print(f"  sklearn time: {sklearn_runtime:.4f} seconds")
            print(f"  custom time:  {custom_runtime:.4f} seconds")
            speedup = sklearn_runtime / custom_runtime
            print(f"  Speedup: {speedup:.2f}x")

        # ====== Equivalence Assessment ======
        is_equivalent = max_diff_overall < 1e-10 and var_diff < 1e-10
        verdict = "PASS ✓" if is_equivalent else "WARN ⚠️"

        if verbose:
            print(f"\n{verdict} EQUIVALENCE VERDICT")
            if is_equivalent:
                print(
                    "The custom PCA implementation is numerically equivalent to "
                    "sklearn's implementation within floating-point precision."
                )
                print("Both produce mathematically identical dimensionality reduction.")
            else:
                print(
                    f"Difference detected ({max_diff_overall:.2e}), but likely due to "
                    f"numerical precision limits or sign conventions."
                )
                print("Results are still practically equivalent for modeling purposes.")

        # ====== Component Analysis ======
        if verbose:
            print("\n(D) COMPONENT ANALYSIS")
            print("-" * 80)
            self._print_component_analysis(sklearn_n_components, custom_n_components)

        # Store comparison results
        self.comparison_results = {
            "sklearn": {
                "n_components": sklearn_n_components,
                "explained_variance_ratio_sum": sklearn_explained_var,
                "runtime": sklearn_runtime,
                "transformed_train": x_tr_pca_sklearn,
                "transformed_val": x_val_pca_sklearn,
                "transformed_test": x_test_pca_sklearn,
            },
            "custom": {
                "n_components": custom_n_components,
                "explained_variance_ratio_sum": custom_explained_var,
                "runtime": custom_runtime,
                "transformed_train": x_tr_pca_custom,
                "transformed_val": x_val_pca_custom,
                "transformed_test": x_test_pca_custom,
            },
            "equivalence": {
                "max_diff_overall": max_diff_overall,
                "var_diff": var_diff,
                "is_equivalent": is_equivalent,
                "verdict": verdict,
            },
        }

        if verbose:
            print("\n" + "=" * 80 + "\n")

        return self.comparison_results

    @staticmethod
    def _print_component_analysis(sklearn_n, custom_n):
        """Print analysis of selected components."""
        print(f"Components selected by sklearn PCA (MLE):  {sklearn_n}")
        print(f"Components selected by custom PCA:         {custom_n}")

        if sklearn_n == custom_n:
            print(f"✓ Component count matches")
        else:
            print(f"⚠️  Component count differs by {abs(sklearn_n - custom_n)}")

        print(
            f"\nNote: MLE (Minka's Maximum Likelihood Estimation) selects the number"
            f"\nof components that best preserves the data manifold structure."
            f"\nTypically retains 90-99% of variance in high-dimensional data."
        )

    def get_transformed_data(self) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Get custom PCA transformed data (preferred for downstream use)."""
        if self.custom_pca is None:
            raise RuntimeError("PCA not fitted yet. Call fit_and_compare() first.")

        return (
            self.comparison_results["custom"]["transformed_train"],
            self.comparison_results["custom"]["transformed_val"],
            self.comparison_results["custom"]["transformed_test"],
        )

    def get_sklearn_transformed_data(self) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Get sklearn PCA transformed data for comparison."""
        if self.sklearn_pca is None:
            raise RuntimeError("PCA not fitted yet. Call fit_and_compare() first.")

        return (
            self.comparison_results["sklearn"]["transformed_train"],
            self.comparison_results["sklearn"]["transformed_val"],
            self.comparison_results["sklearn"]["transformed_test"],
        )

    def print_detailed_component_report(self):
        """Print detailed report on explained variance per component."""
        if self.sklearn_pca is None:
            print("PCA not fitted yet")
            return

        print("\n" + "=" * 80)
        print("DETAILED COMPONENT VARIANCE REPORT")
        print("=" * 80)

        sklearn_var_ratio = self.sklearn_pca.explained_variance_ratio_
        custom_var_ratio = self.custom_pca.explained_variance_ratio_

        print("\nTop 15 Components by Explained Variance")
        print("-" * 80)
        print(
            f"{'PC':<4} | {'sklearn':<12} | {'custom':<12} | {'diff':<12} | "
            f"{'Cumulative (sklearn)':<20}"
        )
        print("-" * 80)

        cumulative_var = 0
        for i in range(min(15, len(sklearn_var_ratio))):
            sk_var = sklearn_var_ratio[i]
            custom_var = custom_var_ratio[i]
            diff = abs(sk_var - custom_var)
            cumulative_var += sk_var

            print(
                f"{i+1:<4} | {sk_var:<12.6f} | {custom_var:<12.6f} | "
                f"{diff:<12.2e} | {cumulative_var:<20.4f}"
            )

        print("\nCumulative Variance Retention")
        print("-" * 80)
        cumsum_sklearn = np.cumsum(sklearn_var_ratio)
        for threshold in [0.80, 0.85, 0.90, 0.95, 0.99]:
            n_components_needed = np.argmax(cumsum_sklearn >= threshold) + 1
            actual_variance = cumsum_sklearn[n_components_needed - 1]
            print(
                f"  {threshold*100:.0f}% variance retained: "
                f"{n_components_needed} components ({actual_variance:.4f})"
            )

        print("\n" + "=" * 80 + "\n")


# ============================================================
# Execute PCA Comparison
# ============================================================

# Initialize comparator
pca_comparator = PCAComparator(random_state=RANDOM_SEED)

# Fit both implementations and compare
comparison_results = pca_comparator.fit_and_compare(
    x_train_scaled=x_tr_scaled,
    x_val_scaled=x_val_scaled,
    x_test_scaled=x_test_scaled,
    n_components="mle",
    verbose=True,
)

# Print detailed component report
pca_comparator.print_detailed_component_report()

# Extract transformed data for downstream use
x_tr_pca_sklearn, x_val_pca_sklearn, x_test_pca_sklearn = (
    pca_comparator.get_sklearn_transformed_data()
)
x_tr_pca_custom, x_val_pca_custom, x_test_pca_custom = (
    pca_comparator.get_transformed_data()
)

# Extract PCA objects for later use (e.g., scree plots)
sklearn_pca = pca_comparator.sklearn_pca
custom_pca = pca_comparator.custom_pca

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"✓ Sklearn PCA: {sklearn_pca.n_components_} components")
print(f"✓ Custom PCA:  {custom_pca.n_components_} components")
print(f"✓ Equivalence: {comparison_results['equivalence']['verdict']}")
print(f"✓ Both implementations ready for downstream modeling")
print("=" * 80 + "\n")

# Store for downstream use
pca_comparison_data = {
    "sklearn_pca": sklearn_pca,
    "custom_pca": custom_pca,
    "x_tr_pca_sklearn": x_tr_pca_sklearn,
    "x_val_pca_sklearn": x_val_pca_sklearn,
    "x_test_pca_sklearn": x_test_pca_sklearn,
    "x_tr_pca_custom": x_tr_pca_custom,
    "x_val_pca_custom": x_val_pca_custom,
    "x_test_pca_custom": x_test_pca_custom,
    "comparison_results": comparison_results,
}

print("Available PCA data for downstream cells:")
for key in pca_comparison_data:
    val = pca_comparison_data[key]
    if isinstance(val, np.ndarray):
        print(f"  • {key}: {val.shape}")
    else:
        print(f"  • {key}: {type(val).__name__}")

In [ ]:
# ============================================================
# Shared Helper Functions & Utilities
# ============================================================

from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_curve,
)

from pathlib import Path

FIGURE_DIR = Path("./figures")
FIGURE_DIR.mkdir(exist_ok=True)


def save_fig(fig, name: str, formats=("pdf", "svg"), dpi: int = 300):
    """Save a matplotlib figure as vector format(s) — no pixelation on zoom/print."""
    for fmt in formats:
        path = FIGURE_DIR / f"{name}.{fmt}"
        fig.savefig(path, format=fmt, dpi=dpi, bbox_inches="tight", facecolor="white")
    print(f"✓ Saved {FIGURE_DIR}/{name}.{{{','.join(formats)}}}")


class ThresholdOptimizer:
    """
    Optimize classification threshold for binary classification.

    Supports multiple optimization criteria:
    - Accuracy (default)
    - F1-score
    - Matthews correlation coefficient
    - Custom metric
    """

    VALID_METRICS = {"accuracy", "f1", "mcc", "youden"}

    def __init__(self, metric: str = "accuracy"):
        """
        Initialize threshold optimizer.

        Parameters
        ----------
        metric : str, default='accuracy'
            Metric to optimize: 'accuracy', 'f1', 'mcc', 'youden'
            - accuracy: (TP + TN) / (TP + TN + FP + FN)
            - f1: Harmonic mean of precision and recall
            - mcc: Matthews Correlation Coefficient (-1 to 1)
            - youden: TPR - FPR (sensitivity - (1-specificity))
        """
        if metric not in self.VALID_METRICS:
            raise ValueError(
                f"metric must be one of {self.VALID_METRICS}, got '{metric}'"
            )
        self.metric = metric

    @staticmethod
    def _youden_j(y_true, y_pred):
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        return sensitivity + specificity - 1

    def _score(self, y_true, y_pred):
        """Dispatch to the configured metric. Kept as a single small mapping
        instead of an if/elif chain so adding a new metric means adding one entry."""
        scorers = {
            "accuracy": lambda: accuracy_score(y_true, y_pred),
            "f1": lambda: f1_score(y_true, y_pred, zero_division=0),
            "mcc": lambda: matthews_corrcoef(y_true, y_pred),
            "youden": lambda: self._youden_j(y_true, y_pred),
        }
        return scorers[self.metric]()

    def find_optimal_threshold(
        self, y_true: np.ndarray, y_proba: np.ndarray, verbose: bool = False
    ) -> Tuple[float, float]:
        """
        Find optimal decision threshold by sweeping through ROC curve.

        Parameters
        ----------
        y_true : np.ndarray
            True binary labels (0 or 1)
        y_proba : np.ndarray
            Predicted probabilities for positive class (shape: n_samples)
        verbose : bool, default=False
            Print threshold search details

        Returns
        -------
        tuple
            (optimal_threshold, metric_value_at_threshold)

        Examples
        --------
        >>> optimizer = ThresholdOptimizer(metric='f1')
        >>> threshold, f1_score = optimizer.find_optimal_threshold(y_true, y_proba)
        """
        _, _, thresholds = roc_curve(y_true, y_proba)

        best_threshold, best_metric_value = 0.5, -np.inf
        for threshold in thresholds:
            y_pred = (y_proba >= threshold).astype(int)
            metric_value = self._score(y_true, y_pred)
            if metric_value > best_metric_value:
                best_threshold, best_metric_value = threshold, metric_value

        if verbose:
            print(f"\nThreshold optimization ({self.metric}):")
            print(f"  Best threshold: {best_threshold:.4f}")
            print(f"  {self.metric.upper()}: {best_metric_value:.4f}")
            print(
                f"  Threshold range: [{thresholds.min():.4f}, {thresholds.max():.4f}]"
            )

        return best_threshold, best_metric_value

    @staticmethod
    def get_threshold_metrics_table(
        y_true: np.ndarray,
        y_proba: np.ndarray,
        thresholds: Optional[List[float]] = None,
    ) -> pd.DataFrame:
        """
        Compute multiple metrics across a range of thresholds.

        Parameters
        ----------
        y_true : np.ndarray
            True binary labels
        y_proba : np.ndarray
            Predicted probabilities
        thresholds : list, optional
            Thresholds to evaluate. Defaults to [0.3, 0.4, 0.5, 0.6, 0.7]

        Returns
        -------
        pd.DataFrame
            Metrics (accuracy, precision, recall, f1) for each threshold
        """
        thresholds = thresholds or [0.3, 0.4, 0.5, 0.6, 0.7]

        results = []
        for thr in thresholds:
            y_pred = (y_proba >= thr).astype(int)
            results.append(
                {
                    "threshold": thr,
                    "accuracy": accuracy_score(y_true, y_pred),
                    "precision": precision_score(y_true, y_pred, zero_division=0),
                    "recall": recall_score(y_true, y_pred, zero_division=0),
                    "f1": f1_score(y_true, y_pred, zero_division=0),
                }
            )

        return pd.DataFrame(results)


# ============================================================
# Model Construction
# ============================================================


def _validate_required_keys(params: Dict, required: set, name: str) -> None:
    """Shared validation used for every model's hyperparameter dict."""
    missing = required - set(params.keys())
    if missing:
        raise ValueError(f"{name} missing keys: {missing}")


def build_models(
    dt_params: Dict,
    rf_params: Dict,
    lr_params: Dict,
    random_state: int = RANDOM_SEED,
) -> Dict[str, object]:
    """
    Build fresh Decision Tree, Random Forest, and Logistic Regression models.

    All hyperparameters are explicitly set (no silent defaults per SonarLint S6973).

    Parameters
    ----------
    dt_params : dict
        Decision Tree hyperparameters: max_depth, min_samples_split,
        min_samples_leaf, criterion
    rf_params : dict
        Random Forest hyperparameters: n_estimators, max_depth,
        min_samples_leaf, max_features
    lr_params : dict
        Logistic Regression hyperparameters: C, max_iter
    random_state : int, default=RANDOM_SEED
        Random state for reproducibility

    Returns
    -------
    dict
        Dictionary of initialized (unfitted) models keyed by name

    Raises
    ------
    ValueError
        If required hyperparameter keys are missing
    """
    _validate_required_keys(
        dt_params,
        {"max_depth", "min_samples_split", "min_samples_leaf", "criterion"},
        "dt_params",
    )
    _validate_required_keys(
        rf_params,
        {"n_estimators", "max_depth", "min_samples_leaf", "max_features"},
        "rf_params",
    )
    _validate_required_keys(lr_params, {"C", "max_iter"}, "lr_params")

    dt = DecisionTreeClassifier(
        max_depth=dt_params["max_depth"],
        min_samples_split=dt_params["min_samples_split"],
        min_samples_leaf=dt_params["min_samples_leaf"],
        criterion=dt_params["criterion"],
        ccp_alpha=0.0,  # No cost complexity pruning
        random_state=random_state,
    )

    rf = RandomForestClassifier(
        n_estimators=rf_params["n_estimators"],
        max_depth=rf_params["max_depth"],
        min_samples_leaf=rf_params["min_samples_leaf"],
        max_features=rf_params["max_features"],
        bootstrap=True,
        oob_score=False,
        n_jobs=-1,
        random_state=random_state,
    )

    # LogisticRegression — 'multi_class' omitted: only relevant for multiclass,
    # binary works fine with just C, solver, max_iter
    lr = LogisticRegression(
        C=lr_params["C"],
        solver="lbfgs",
        max_iter=lr_params["max_iter"],
        random_state=random_state,
    )

    return {
        "Decision Tree": dt,
        "Random Forest": rf,
        "Logistic Regression": lr,
    }


# ============================================================
# Binary Classification Experiment
# ============================================================


def _evaluate_binary_model(model, x_val, y_val, x_test, y_test, threshold_optimizer):
    """Fit is assumed already done. Tune threshold on val, score on test."""
    y_val_proba = model.predict_proba(x_val)[:, 1]
    optimal_threshold, _ = threshold_optimizer.find_optimal_threshold(
        y_val, y_val_proba, verbose=False
    )

    y_test_proba = model.predict_proba(x_test)[:, 1]
    y_test_pred = (y_test_proba >= optimal_threshold).astype(int)

    cm = confusion_matrix(y_test, y_test_pred)
    tn, fp, fn, tp = cm.ravel()

    return {
        "threshold": optimal_threshold,
        "accuracy": accuracy_score(y_test, y_test_pred),
        "recall": recall_score(y_test, y_test_pred),
        "precision": precision_score(y_test, y_test_pred),
        "f1": f1_score(y_test, y_test_pred),
        "sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "confusion_matrix": cm,
        "y_test_proba": y_test_proba,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


def _print_binary_results(model_name, eval_result):
    print(f"  Optimal threshold: {eval_result['threshold']:.4f}")
    print(
        f"  Accuracy:  {eval_result['accuracy']:.4f} ({eval_result['accuracy'] * 100:.2f}%)"
    )
    print(f"  Recall:    {eval_result['recall']:.4f} (Sensitivity)")
    print(f"  Precision: {eval_result['precision']:.4f}")
    print(f"  F1-Score:  {eval_result['f1']:.4f}")
    print(f"  Specificity: {eval_result['specificity']:.4f}")
    print(f"\n  Confusion Matrix:")
    print(f"    TN={eval_result['tn']:6d}  FP={eval_result['fp']:6d}")
    print(f"    FN={eval_result['fn']:6d}  TP={eval_result['tp']:6d}")


def run_binary_experiment(
    models: Dict[str, object],
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_val: np.ndarray,
    y_val: np.ndarray,
    x_test: np.ndarray,
    y_test: np.ndarray,
    variant_label: str,
    threshold_metric: str = "accuracy",
) -> List[Dict]:
    """
    Fit binary classification models and evaluate with threshold optimization.

    Workflow:
    1. Fit each model on training data
    2. Find optimal threshold on validation data
    3. Evaluate on test data with optimized threshold

    Parameters
    ----------
    models : dict
        Dictionary of sklearn model instances (unfitted)
    x_train, y_train : array-like
        Training features and binary labels
    x_val, y_val : array-like
        Validation features and labels (for threshold tuning)
    x_test, y_test : array-like
        Test features and labels (for final evaluation)
    variant_label : str
        Descriptive label for this experiment (e.g., "No PCA", "PCA")
    threshold_metric : str, default='accuracy'
        Metric to optimize threshold by: 'accuracy', 'f1', 'mcc', 'youden'

    Returns
    -------
    list
        List of result dictionaries, one per model, with keys:
        Model, Variant, Accuracy, Recall, Precision, F1, _fitted_model, _confusion_matrix
    """
    print(f"\n{'=' * 70}")
    print(f"BINARY CLASSIFICATION — {variant_label}")
    print(f"{'=' * 70}")

    threshold_optimizer = ThresholdOptimizer(metric=threshold_metric)
    results = []

    for model_name, model in models.items():
        print(f"\n{model_name}")
        print("-" * 70)

        model.fit(x_train, y_train)
        eval_result = _evaluate_binary_model(
            model, x_val, y_val, x_test, y_test, threshold_optimizer
        )
        _print_binary_results(model_name, eval_result)

        results.append(
            {
                "Model": model_name,
                "Variant": variant_label,
                "Threshold": eval_result["threshold"],
                "Accuracy": eval_result["accuracy"],
                "Recall": eval_result["recall"],
                "Precision": eval_result["precision"],
                "F1": eval_result["f1"],
                "Sensitivity": eval_result["sensitivity"],
                "Specificity": eval_result["specificity"],
                "_fitted_model": model,
                "_confusion_matrix": eval_result["confusion_matrix"],
                "_y_test_proba": eval_result["y_test_proba"],
            }
        )

    print(f"\n{'=' * 70}\n")
    return results


# ============================================================
# Multiclass Classification Experiment
# ============================================================


def _print_multiclass_results(acc, report, class_names):
    print(f"  Overall Accuracy: {acc:.4f} ({acc * 100:.2f}%)")

    print(f"\n  MACRO AVERAGES (equal weight to all classes)")
    print(f"    Precision: {report['macro avg']['precision']:.4f}")
    print(f"    Recall:    {report['macro avg']['recall']:.4f}")
    print(f"    F1-Score:  {report['macro avg']['f1-score']:.4f}")

    print(f"\n  WEIGHTED AVERAGES (weighted by class support)")
    print(f"    Precision: {report['weighted avg']['precision']:.4f}")
    print(f"    Recall:    {report['weighted avg']['recall']:.4f}")
    print(f"    F1-Score:  {report['weighted avg']['f1-score']:.4f}")

    print(f"\n  PER-CLASS METRICS")
    print(
        f"  {'Class':<12} | {'Precision':>10} | {'Recall':>10} | {'F1-Score':>10} | {'Support':>10}"
    )
    print(f"  {'-' * 12}-+-{'-' * 10}-+-{'-' * 10}-+-{'-' * 10}-+-{'-' * 10}")
    for cls_name in class_names:
        cls_report = report[cls_name]
        print(
            f"  {cls_name:<12} | {cls_report['precision']:10.4f} | {cls_report['recall']:10.4f} | "
            f"{cls_report['f1-score']:10.4f} | {int(cls_report['support']):10d}"
        )


def run_multiclass_experiment(
    models: Dict[str, object],
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_test: np.ndarray,
    y_test: np.ndarray,
    class_names: np.ndarray,
    variant_label: str,
) -> List[Dict]:
    """
    Fit multiclass classification models and evaluate.

    Parameters
    ----------
    models : dict
        Dictionary of sklearn model instances (unfitted)
    x_train, y_train : array-like
        Training features and multiclass labels (0, 1, 2, ..., n_classes-1)
    x_test, y_test : array-like
        Test features and labels
    class_names : array-like
        Human-readable class names (e.g., ['Normal', 'DoS', 'Probe', 'R2L', 'U2R'])
    variant_label : str
        Descriptive label for this experiment (e.g., "No PCA", "PCA")

    Returns
    -------
    list
        List of result dictionaries with macro/weighted averages and per-class metrics
    """
    print(f"\n{'=' * 70}")
    print(f"MULTICLASS CLASSIFICATION — {variant_label}")
    print(f"{'=' * 70}")

    results = []

    for model_name, model in models.items():
        print(f"\n{model_name}")
        print("-" * 70)

        model.fit(x_train, y_train)
        y_test_pred = model.predict(x_test)
        acc = accuracy_score(y_test, y_test_pred)

        report = classification_report(
            y_test,
            y_test_pred,
            target_names=class_names,
            output_dict=True,
            zero_division=0,
        )
        _print_multiclass_results(acc, report, class_names)

        results.append(
            {
                "Model": model_name,
                "Variant": variant_label,
                "Accuracy": acc,
                "Macro Precision": report["macro avg"]["precision"],
                "Macro Recall": report["macro avg"]["recall"],
                "Macro F1": report["macro avg"]["f1-score"],
                "Weighted Precision": report["weighted avg"]["precision"],
                "Weighted Recall": report["weighted avg"]["recall"],
                "Weighted F1": report["weighted avg"]["f1-score"],
                "_per_class_report": report,
                "_fitted_model": model,
                "_y_test_pred": y_test_pred,
            }
        )

    print(f"\n{'=' * 70}\n")
    return results


# ============================================================
# Initialize Visualizer
# ============================================================


class ModelVisualizer:
    """Generate standardized visualizations for model evaluation."""

    @staticmethod
    def _top_n_indices(scores: np.ndarray, top_n: int) -> np.ndarray:
        return np.argsort(scores)[-top_n:][::-1]

    @staticmethod
    def _plot_horizontal_importance(
        scores, feature_names, title, xlabel, figsize, top_n, save_name=None
    ):
        feature_names = np.asarray(feature_names)
        top_indices = ModelVisualizer._top_n_indices(scores, top_n)

        fig, ax = plt.subplots(figsize=figsize)
        ax.barh(range(len(top_indices)), scores[top_indices], align="center")
        ax.set_yticks(range(len(top_indices)))
        ax.set_yticklabels(feature_names[top_indices])
        ax.set_xlabel(xlabel)
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.grid(axis="x", alpha=0.3)

        plt.tight_layout()
        if save_name:
            save_fig(fig, save_name)
        plt.show()

    @staticmethod
    def plot_feature_importance(
        model, feature_names, title, top_n=15, figsize=(10, 8), save_name=None
    ):
        if not hasattr(model, "feature_importances_"):
            raise ValueError(
                f"Model {type(model).__name__} has no feature_importances_ attribute"
            )
        ModelVisualizer._plot_horizontal_importance(
            model.feature_importances_,
            feature_names,
            title,
            "Feature Importance",
            figsize,
            top_n,
            save_name,
        )

    @staticmethod
    def plot_logreg_coefficients(
        model, feature_names, title, top_n=15, figsize=(10, 8), save_name=None
    ):
        if not hasattr(model, "coef_"):
            raise ValueError(f"Model {type(model).__name__} has no coef_ attribute")
        ModelVisualizer._plot_horizontal_importance(
            np.abs(model.coef_[0]),
            feature_names,
            title,
            "|Coefficient| (standardized features)",
            figsize,
            top_n,
            save_name,
        )

    @staticmethod
    def plot_confusion_matrix(
        y_true, y_pred, labels, title, figsize=(8, 8), cmap="Blues", save_name=None
    ):
        cm = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)

        fig, ax = plt.subplots(figsize=figsize)
        ax.grid(False)
        disp.plot(ax=ax, cmap=cmap, colorbar=True)
        ax.set_title(title, fontsize=12, fontweight="bold")

        plt.tight_layout()
        if save_name:
            save_fig(fig, save_name)
        plt.show()


# ============================================================
# Initialize Visualizer
# ============================================================

visualizer = ModelVisualizer()

print("=" * 80)
print("HELPER UTILITIES INITIALIZED")
print("=" * 80)
print("\nAvailable utilities:")
print("  • ThresholdOptimizer: Optimize decision thresholds by multiple metrics")
print("  • build_models(): Instantiate DT, RF, LR with explicit hyperparameters")
print("  • run_binary_experiment(): Fit & evaluate binary classifiers")
print("  • run_multiclass_experiment(): Fit & evaluate multiclass classifiers")
print("  • ModelVisualizer: Plot feature importance, coefficients, confusion matrices")
print("=" * 80 + "\n")

In [ ]:
# ============================================================
# Hyperparameter Tuning with Comprehensive Diagnostics
# ============================================================
# Hyperparameters are tuned ONLY on binary training data.
# Best params are reused for:
#   • Sklearn PCA models
#   • Custom PCA models
#   • Multiclass experiments
#
# F1-score used as optimization metric (class imbalance).
# ============================================================

import json
from datetime import datetime
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold


class HyperparameterTuner:
    """
    Comprehensive hyperparameter tuning with diagnostics.

    Features:
    - Stratified K-fold cross-validation
    - Randomized search for efficiency
    - Detailed logging of search process
    - Comparison of candidate models
    - Export of best parameters
    """

    def __init__(
        self, cv_splits: int = 3, random_state: int = RANDOM_SEED, n_jobs: int = -1
    ):
        """
        Initialize tuner.

        Parameters
        ----------
        cv_splits : int, default=3
            Number of stratified K-fold splits
        random_state : int, default=RANDOM_SEED
            Random state for reproducibility
        n_jobs : int, default=-1
            Number of parallel jobs (-1 = all cores)
        """
        self.cv_splits = cv_splits
        self.random_state = random_state
        self.n_jobs = n_jobs
        self.cv_splitter = StratifiedKFold(
            n_splits=cv_splits, shuffle=True, random_state=random_state
        )
        self.search_results = {}

    def tune(
        self,
        model_name: str,
        estimator: object,
        param_grid: Dict,
        x_train: np.ndarray,
        y_train: np.ndarray,
        n_iter: int = 10,
        scoring: str = "f1",
        verbose: bool = True,
    ) -> object:
        """
        Perform randomized hyperparameter search.

        Parameters
        ----------
        model_name : str
            Name of the model (for logging)
        estimator : sklearn estimator
            Base machine learning model (unfitted)
        param_grid : dict
            Search space for hyperparameters
        x_train : np.ndarray
            Training features
        y_train : np.ndarray
            Training labels (binary: 0 or 1)
        n_iter : int, default=10
            Number of random parameter combinations to try
        scoring : str, default='f1'
            Metric to optimize: 'f1', 'accuracy', 'roc_auc', 'precision', 'recall'
        verbose : bool, default=True
            Print search progress and results

        Returns
        -------
        RandomizedSearchCV
            Fitted search object with best_params_, best_score_, cv_results_

        Raises
        ------
        ValueError
            If param_grid is empty
        """
        if not param_grid:
            raise ValueError("param_grid cannot be empty")

        if verbose:
            print(f"\n{'=' * 80}")
            print(f"HYPERPARAMETER TUNING — {model_name}")
            print(f"{'=' * 80}")
            print(f"Estimator: {type(estimator).__name__}")
            print(f"Scoring metric: {scoring}")
            print(f"CV splits: {self.cv_splits}")
            print(f"Search iterations: {n_iter}")
            print(f"Parameter grid size: {self._estimate_param_space(param_grid)}")

        start_time = datetime.now()
        search = RandomizedSearchCV(
            estimator=estimator,
            param_distributions=param_grid,
            n_iter=n_iter,
            cv=self.cv_splitter,
            scoring=scoring,
            random_state=self.random_state,
            n_jobs=self.n_jobs,
            verbose=0,
        )
        search.fit(x_train, y_train)
        elapsed = (datetime.now() - start_time).total_seconds()

        self.search_results[model_name] = {
            "search": search,
            "elapsed_seconds": elapsed,
            "scoring": scoring,
        }

        if verbose:
            self._print_search_summary(search, elapsed)

        return search

    @staticmethod
    def _estimate_param_space(param_grid: Dict) -> int:
        """Estimate total hyperparameter space size."""
        space = 1
        for values in param_grid.values():
            space *= len(values)
        return space

    @staticmethod
    def _print_search_summary(search: object, elapsed: float):
        """Print detailed search summary."""
        print(f"\nSearch Results")
        print("-" * 80)
        print(f"Best {search.scoring} score: {search.best_score_:.4f}")
        print(f"Search time: {elapsed:.2f} seconds")
        print(f"\nBest Hyperparameters:")
        for param, value in search.best_params_.items():
            print(f"  {param:<25}: {value}")

    def _get_result(self, model_name: str) -> Dict:
        """Shared lookup + not-found guard for the result-accessor methods below."""
        if model_name not in self.search_results:
            raise ValueError(f"Model '{model_name}' not found in search results")
        return self.search_results[model_name]

    def get_best_params(self, model_name: str) -> Dict:
        """Get best parameters for a tuned model."""
        return self._get_result(model_name)["search"].best_params_

    def get_best_score(self, model_name: str) -> float:
        """Get best CV score for a tuned model."""
        return self._get_result(model_name)["search"].best_score_

    def print_all_results(self):
        """Print summary of all tuned models."""
        print("\n" + "=" * 80)
        print("HYPERPARAMETER TUNING SUMMARY — ALL MODELS")
        print("=" * 80)

        for model_name, result in self.search_results.items():
            search = result["search"]
            print(f"\n{model_name}")
            print("-" * 80)
            print(f"Best {result['scoring']}: {search.best_score_:.4f}")
            print(f"Time: {result['elapsed_seconds']:.2f} seconds")
            print("Best parameters:")
            for param, value in search.best_params_.items():
                print(f"  {param:<25}: {value}")

    def compare_cv_folds(self, model_name: str, top_n: int = 5):
        """Show performance across CV folds for top parameter sets."""
        search = self._get_result(model_name)["search"]
        cv_results = pd.DataFrame(search.cv_results_)
        top_results = cv_results.sort_values("mean_test_score", ascending=False).head(
            top_n
        )

        print(f"\n{model_name} — Top {top_n} Parameter Sets (by CV score)")
        print("-" * 80)

        for rank, (_, row) in enumerate(top_results.iterrows(), start=1):
            print(
                f"\nRank {rank}: Mean score = {row['mean_test_score']:.4f} (std: {row['std_test_score']:.4f})"
            )
            params = {
                k.replace("param_", ""): row[k]
                for k in row.index
                if k.startswith("param_")
            }
            for param, value in params.items():
                print(f"  {param:<25}: {value}")

    def export_best_params(self, output_path: str = None) -> Dict:
        """
        Export best parameters for all tuned models.

        Parameters
        ----------
        output_path : str, optional
            Path to save as JSON. If None, just return dict.

        Returns
        -------
        dict
            Best parameters keyed by model name
        """
        best_params_dict = {
            name: self.get_best_params(name) for name in self.search_results
        }

        if output_path:
            output_path = Path(output_path)
            output_path.parent.mkdir(parents=True, exist_ok=True)
            with open(output_path, "w") as f:
                json.dump(best_params_dict, f, indent=2)
            print(f"✓ Best parameters saved to: {output_path}")

        return best_params_dict


# ============================================================
# Parameter Grids & Model Specs
# ============================================================

# Each entry drives one tune() call: estimator, its search space, and how many
# random combinations to sample. Keeping these together avoids repeating the
# same "build estimator -> call tune() -> verbose=True" block three times.
MODEL_SPECS = [
    {
        "name": "Decision Tree",
        "estimator": DecisionTreeClassifier(ccp_alpha=0.0, random_state=RANDOM_SEED),
        "param_grid": {
            "max_depth": [None, 10, 15, 20, 25, 30],
            "min_samples_split": [2, 5, 10, 20],
            "min_samples_leaf": [1, 2, 4, 8],
            "criterion": ["gini", "entropy"],
        },
        "n_iter": 15,  # 15 random combos out of the full grid
    },
    {
        "name": "Random Forest",
        "estimator": RandomForestClassifier(
            bootstrap=True, random_state=RANDOM_SEED, n_jobs=-1
        ),
        "param_grid": {
            "n_estimators": [100, 150],
            "max_depth": [None, 15, 25],
            "min_samples_leaf": [1, 2, 4],
            "max_features": ["sqrt", "log2"],
        },
        "n_iter": 6,  # 2*3*3*2 = 36 total combos, 6/36 ≈ 17%
    },
    {
        "name": "Logistic Regression",
        "estimator": LogisticRegression(
            solver="lbfgs", warm_start=False, random_state=RANDOM_SEED
        ),
        "param_grid": {
            "C": np.logspace(-3, 3, 13),  # inverse regularization strength
            "max_iter": [500, 1000, 2000],
        },
        "n_iter": 8,  # 13*3 = 39 total combos, 8/39 ≈ 20%
    },
]


def run_hyperparameter_tuning(
    x_tr_scaled, y_tr, output_path=Path("./best_hyperparameters.json")
):
    """
    Run randomized search for every model in MODEL_SPECS, print diagnostics,
    and export the best parameters.

    Returns
    -------
    dict with keys: tuner, searches (dict keyed by model name), BEST_PARAMS
    """
    print("=" * 80)
    print("HYPERPARAMETER TUNING")
    print("=" * 80)
    print("\nConfiguration:")
    print(
        f"  • Training data: {len(y_tr):,} samples (binary labels: 0=Normal, 1=Attack)"
    )
    print(f"  • CV strategy: Stratified 3-fold")
    print(f"  • Scoring metric: F1-score (accounts for class imbalance)")
    print(f"  • Search method: Randomized Search (efficient)\n")

    tuner = HyperparameterTuner(cv_splits=3, random_state=RANDOM_SEED, n_jobs=-1)

    searches = {
        spec["name"]: tuner.tune(
            model_name=spec["name"],
            estimator=spec["estimator"],
            param_grid=spec["param_grid"],
            x_train=x_tr_scaled,
            y_train=y_tr,
            n_iter=spec["n_iter"],
            scoring="f1",
            verbose=True,
        )
        for spec in MODEL_SPECS
    }

    tuner.print_all_results()

    print("\n" + "=" * 80)
    print("TOP 5 PARAMETER SETS — Cross-Validation Performance")
    print("=" * 80)
    for spec in MODEL_SPECS:
        tuner.compare_cv_folds(spec["name"], top_n=5)

    # Export once (also writes to disk); reuse the same dict for the summary table
    best_params = tuner.export_best_params(output_path=output_path)

    print("\n" + "=" * 80)
    print("BEST HYPERPARAMETERS — FINAL SELECTION")
    print("=" * 80)
    best_params_table = pd.DataFrame(best_params).T
    print("\n")
    print(best_params_table.to_string())

    print("\n" + "=" * 80)
    print("✓ Hyperparameter tuning complete")
    print("✓ Best parameters extracted for downstream experiments")
    print("=" * 80 + "\n")

    return {"tuner": tuner, "searches": searches, "BEST_PARAMS": best_params}


tuning_result = run_hyperparameter_tuning(x_tr_scaled, y_tr)

tuner = tuning_result["tuner"]
BEST_PARAMS = tuning_result["BEST_PARAMS"]
dt_search = tuning_result["searches"]["Decision Tree"]
rf_search = tuning_result["searches"]["Random Forest"]
lr_search = tuning_result["searches"]["Logistic Regression"]
dt_params = BEST_PARAMS["Decision Tree"]
rf_params = BEST_PARAMS["Random Forest"]
lr_params = BEST_PARAMS["Logistic Regression"]

hyperparameter_tuning_data = {
    "tuner": tuner,
    "dt_search": dt_search,
    "rf_search": rf_search,
    "lr_search": lr_search,
    "BEST_PARAMS": BEST_PARAMS,
    "dt_params": dt_params,
    "rf_params": rf_params,
    "lr_params": lr_params,
}

print("Available for downstream cells:")
for key in hyperparameter_tuning_data:
    print(f"  • {key}")

In [ ]:
# ============================================================
# Binary Classification Experiments — Multiple Feature Spaces
# ============================================================
# Compare model performance across:
#   1. No PCA (original 41 features)
#   2. PCA (sklearn) — ~40 components, 100% variance
#   3. PCA (from-scratch) — custom NumPy implementation
#
# Each variant uses the same best hyperparameters (tuned
# on the No-PCA training data to avoid circular optimization).
# ============================================================

from datetime import datetime
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

METRICS = ["Accuracy", "Precision", "Recall", "F1", "Sensitivity", "Specificity"]


class BinaryExperimentRunner:
    """
    Orchestrate binary classification experiments across feature spaces.

    Manages:
    - Multiple feature space variants
    - Model instantiation with tuned hyperparameters
    - Result collection and comparison
    - Experiment tracking
    """

    def __init__(self, random_state: int = RANDOM_SEED):
        """Initialize experiment runner."""
        self.random_state = random_state
        self.results = []
        self.experiment_log = []

    def run_all_variants(
        self,
        feature_variants: Dict[str, Tuple],
        y_train: np.ndarray,
        y_val: np.ndarray,
        y_test: np.ndarray,
        best_params: Dict,
        threshold_metric: str = "accuracy",
        verbose: bool = True,
    ) -> List[Dict]:
        """
        Run binary classification across all feature space variants.

        Parameters
        ----------
        feature_variants : dict
            Mapping of variant names to (x_train, x_val, x_test) tuples
            Example:
            {
                "No PCA": (x_tr_scaled, x_val_scaled, x_test_scaled),
                "PCA (sklearn)": (x_tr_pca_sklearn, x_val_pca_sklearn, x_test_pca_sklearn),
                "PCA (custom)": (x_tr_pca_custom, x_val_pca_custom, x_test_pca_custom),
            }
        y_train, y_val, y_test : np.ndarray
            Binary labels for each set
        best_params : dict
            Best hyperparameters for each model:
            {
                "Decision Tree": {...},
                "Random Forest": {...},
                "Logistic Regression": {...},
            }
        threshold_metric : str, default='accuracy'
            Metric to optimize thresholds: 'accuracy', 'f1', 'mcc', 'youden'
        verbose : bool, default=True
            Print detailed results for each variant

        Returns
        -------
        list
            List of result dictionaries from all experiments
        """
        if verbose:
            print("\n" + "=" * 90)
            print("BINARY CLASSIFICATION EXPERIMENTS")
            print("=" * 90)
            print(f"\nConfiguration:")
            print(f"  • Feature space variants: {len(feature_variants)}")
            print(f"  • Training samples: {len(y_train):,}")
            print(f"  • Validation samples: {len(y_val):,}")
            print(f"  • Test samples: {len(y_test):,}")
            print(
                f"  • Class balance (train): {(y_train == 0).sum():,} / {(y_train == 1).sum():,}"
            )
            print(f"  • Threshold metric: {threshold_metric}")
            print()

        self.results = []

        for variant_idx, (variant_label, feature_data) in enumerate(
            feature_variants.items(), 1
        ):
            x_train_v, x_val_v, x_test_v = feature_data

            if verbose:
                print(f"\n[{variant_idx}/{len(feature_variants)}] {variant_label}")
                print(f"  Feature shape: {x_train_v.shape}")

            models = build_models(
                dt_params=best_params["Decision Tree"],
                rf_params=best_params["Random Forest"],
                lr_params=best_params["Logistic Regression"],
                random_state=self.random_state,
            )

            variant_results = run_binary_experiment(
                models=models,
                x_train=x_train_v,
                y_train=y_train,
                x_val=x_val_v,
                y_val=y_val,
                x_test=x_test_v,
                y_test=y_test,
                variant_label=variant_label,
                threshold_metric=threshold_metric,
            )
            self.results.extend(variant_results)

            self.experiment_log.append(
                {
                    "variant": variant_label,
                    "n_features": x_train_v.shape[1],
                    "n_models": len(models),
                    "completed_at": datetime.now(),
                }
            )

        if verbose:
            print("\n" + "=" * 90)

        return self.results

    def create_comparison_dataframe(self) -> pd.DataFrame:
        """
        Create comparison DataFrame from all experiments.

        Returns
        -------
        pd.DataFrame
            Results with columns: Model, Variant, Accuracy, Recall, Precision, F1, ...
        """
        return pd.DataFrame(self.results)

    def _results_or_warn(self) -> pd.DataFrame | None:
        """Shared guard: return the results dataframe, or None + a message if empty."""
        if not self.results:
            print("No results available")
            return None
        return self.create_comparison_dataframe()

    def print_accuracy_pivot(self):
        """Print accuracy comparison across models and variants."""
        results_df = self._results_or_warn()
        if results_df is None:
            return

        print("\n" + "=" * 90)
        print("ACCURACY COMPARISON ACROSS VARIANTS")
        print("=" * 90)

        accuracy_pivot_pct = (
            results_df.pivot(index="Model", columns="Variant", values="Accuracy") * 100
        ).round(2)

        print("\nAccuracy (%):\n")
        print(accuracy_pivot_pct.to_string())

        if "No PCA" in accuracy_pivot_pct.columns:
            print("\n\nImprovement over 'No PCA' baseline:")
            baseline = accuracy_pivot_pct["No PCA"]
            for col in accuracy_pivot_pct.columns:
                if col == "No PCA":
                    continue
                improvement = accuracy_pivot_pct[col] - baseline
                print(f"\n{col}:")
                for model, delta in improvement.items():
                    sign = "+" if delta > 0 else ""
                    print(f"  {model:<25}: {sign}{delta:6.2f}%")

        print("\n" + "=" * 90)

    @staticmethod
    def _print_metrics_table(df: pd.DataFrame, row_label_col: str, header_label: str):
        """Shared tabular renderer: one row per record, columns = METRICS.
        Used by both print_detailed_results (rows=Model) and
        compare_variants_for_model (rows=Variant)."""
        print(f"{header_label:<25} | " + " | ".join(f"{m:>10}" for m in METRICS))
        print("-" * 90)
        for _, row in df.iterrows():
            values = " | ".join(f"{row[m]:>10.4f}" for m in METRICS)
            print(f"{row[row_label_col]:<25} | {values}")

    def print_detailed_results(self):
        """Print comprehensive results for all variants and models."""
        results_df = self._results_or_warn()
        if results_df is None:
            return

        print("\n" + "=" * 90)
        print("DETAILED RESULTS — ALL METRICS")
        print("=" * 90)

        for variant in results_df["Variant"].unique():
            print(f"\n{variant}")
            print("-" * 90)
            variant_data = results_df[results_df["Variant"] == variant]
            self._print_metrics_table(
                variant_data, row_label_col="Model", header_label="Model"
            )

        print("\n" + "=" * 90)

    def print_best_models(self, metric: str = "F1"):
        """
        Identify and print best models by metric.

        Parameters
        ----------
        metric : str, default='F1'
            Metric to rank by
        """
        results_df = self._results_or_warn()
        if results_df is None:
            return

        print(f"\n" + "=" * 90)
        print(f"BEST MODELS BY {metric}")
        print("=" * 90)

        top_models = results_df.nlargest(5, metric)[
            ["Model", "Variant", metric, "Accuracy", "Recall"]
        ]

        for rank, (_, row) in enumerate(top_models.iterrows(), 1):
            print(
                f"\n{rank}. {row['Model']} ({row['Variant']})"
                f"\n   {metric}: {row[metric]:.4f}, Accuracy: {row['Accuracy']:.4f}, "
                f"Recall: {row['Recall']:.4f}"
            )

        print("\n" + "=" * 90)

    def get_variant_data(self, variant_name: str) -> pd.DataFrame:
        """Get results for a specific variant."""
        return self.create_comparison_dataframe().pipe(
            lambda df: df[df["Variant"] == variant_name]
        )

    def compare_variants_for_model(self, model_name: str):
        """Print comparison of all variants for a specific model."""
        results_df = self._results_or_warn()
        if results_df is None:
            return

        model_data = results_df[results_df["Model"] == model_name]

        print(f"\n" + "=" * 90)
        print(f"{model_name} — VARIANT COMPARISON")
        print("=" * 90)

        self._print_metrics_table(
            model_data, row_label_col="Variant", header_label="Variant"
        )

        print("\n" + "=" * 90)


# ============================================================
# Reporting helpers (script section)
# ============================================================


def _print_group_stats(
    results_df: pd.DataFrame, group_col: str, value_col: str = "Accuracy"
):
    """Print mean/std/min/max of value_col, grouped by group_col.
    Shared by the by-variant and by-model summaries below (identical shape,
    only the grouping column differs)."""
    for group_value in results_df[group_col].unique():
        subset = results_df[results_df[group_col] == group_value][value_col]
        print(
            f"\n{group_value}:"
            f"\n  Mean:   {subset.mean():.4f}"
            f"\n  Std:    {subset.std():.4f}"
            f"\n  Min:    {subset.min():.4f}"
            f"\n  Max:    {subset.max():.4f}"
        )


def _print_statistical_summary(results_df: pd.DataFrame):
    print("\n" + "=" * 90)
    print("STATISTICAL SUMMARY")
    print("=" * 90)

    print("\nAccuracy statistics by variant:")
    _print_group_stats(results_df, group_col="Variant")

    print("\n\nAccuracy statistics by model:")
    _print_group_stats(results_df, group_col="Model")

    print("\n" + "=" * 90)
    print("✓ Binary classification experiments complete")
    print("=" * 90 + "\n")


# ============================================================
# Execute Binary Experiments
# ============================================================


def run_binary_experiments_pipeline(
    x_tr_scaled,
    x_val_scaled,
    x_test_scaled,
    x_tr_pca_sklearn,
    x_val_pca_sklearn,
    x_test_pca_sklearn,
    x_tr_pca_custom,
    x_val_pca_custom,
    x_test_pca_custom,
    y_tr,
    y_val,
    y_test,
    best_params,
):
    """Run the full binary-classification comparison across feature-space variants
    and print all standard reports. Returns everything downstream cells need."""
    print("\n" + "=" * 90)
    print("PREPARING BINARY CLASSIFICATION EXPERIMENTS")
    print("=" * 90)

    binary_variants = {
        "No PCA": (x_tr_scaled, x_val_scaled, x_test_scaled),
        "PCA (sklearn)": (x_tr_pca_sklearn, x_val_pca_sklearn, x_test_pca_sklearn),
        "PCA (custom)": (x_tr_pca_custom, x_val_pca_custom, x_test_pca_custom),
    }

    print(f"\nFeature space variants:")
    for variant_name, (x_tr_v, _, _) in binary_variants.items():
        print(f"  • {variant_name:<20}: {x_tr_v.shape[1]:3d} features")

    binary_runner = BinaryExperimentRunner(random_state=RANDOM_SEED)
    binary_results = binary_runner.run_all_variants(
        feature_variants=binary_variants,
        y_train=y_tr,
        y_val=y_val,
        y_test=y_test,
        best_params=best_params,
        threshold_metric="accuracy",
        verbose=True,
    )

    binary_runner.print_accuracy_pivot()
    binary_runner.print_detailed_results()
    binary_runner.print_best_models(metric="F1")

    print("\n" + "=" * 90)
    print("MODEL-SPECIFIC VARIANT ANALYSIS")
    print("=" * 90)
    for model_name in ["Decision Tree", "Random Forest", "Logistic Regression"]:
        binary_runner.compare_variants_for_model(model_name)

    results_df = binary_runner.create_comparison_dataframe()
    _print_statistical_summary(results_df)

    binary_experiment_data = {
        "runner": binary_runner,
        "binary_results": binary_results,
        "results_df": results_df,
        "binary_variants": binary_variants,
    }

    print("Available for downstream cells:")
    for key, val in binary_experiment_data.items():
        if isinstance(val, pd.DataFrame):
            print(f"  • {key}: {val.shape}")
        elif isinstance(val, list):
            print(f"  • {key}: {len(val)} items")
        else:
            print(f"  • {key}: {type(val).__name__}")

    return binary_experiment_data


binary_experiment_data = run_binary_experiments_pipeline(
    x_tr_scaled,
    x_val_scaled,
    x_test_scaled,
    x_tr_pca_sklearn,
    x_val_pca_sklearn,
    x_test_pca_sklearn,
    x_tr_pca_custom,
    x_val_pca_custom,
    x_test_pca_custom,
    y_tr,
    y_val,
    y_test,
    BEST_PARAMS,
)

binary_runner = binary_experiment_data["runner"]
binary_results = binary_experiment_data["binary_results"]
results_df = binary_experiment_data["results_df"]
binary_variants = binary_experiment_data["binary_variants"]

In [ ]:
# ============================================================
# Post-Experiment Analysis & Visualization
# ============================================================

from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import auc, roc_curve

METRICS = ["Accuracy", "Precision", "Recall", "F1", "Sensitivity", "Specificity"]
MODEL_ORDER = ["Decision Tree", "Random Forest", "Logistic Regression"]
VARIANT_COLORS = ["#2ca02c", "#ff7f0e", "#1f77b4"]
MODEL_COLORS = {
    "Decision Tree": "#1f77b4",
    "Random Forest": "#d62728",
    "Logistic Regression": "#2ca02c",
}


class BinaryResultsAnalyzer:
    """
    Comprehensive analysis and visualization of binary classification results.

    Features:
    - Accuracy comparison tables
    - Feature importance visualization
    - Confusion matrix analysis
    - Performance metrics summary
    """

    def __init__(self, binary_results: List[Dict], feature_names: List[str]):
        """
        Initialize analyzer.

        Parameters
        ----------
        binary_results : list
            Results from binary experiments
        feature_names : list
            Names of original features (for importance plots)
        """
        self.results_df = pd.DataFrame(binary_results)
        self.feature_names = np.array(feature_names)
        self.binary_results = binary_results

    # ------------------------------------------------------------------
    # Tables
    # ------------------------------------------------------------------

    def print_accuracy_comparison(
        self, include_variants: List[str] = None, as_percentage: bool = True
    ):
        """
        Print accuracy comparison table.

        Parameters
        ----------
        include_variants : list, optional
            Variants to include (default: all from data)
        as_percentage : bool, default=True
            Display as percentage or decimal
        """
        pivot = self.results_df.pivot(
            index="Model", columns="Variant", values="Accuracy"
        )

        if include_variants:
            available = [v for v in include_variants if v in pivot.columns]
            pivot = pivot[available]

        if as_percentage:
            pivot, unit = (pivot * 100).round(2), "%"
        else:
            pivot, unit = pivot.round(4), ""

        print("\n" + "=" * 80)
        print("ACCURACY COMPARISON — BINARY CLASSIFICATION")
        print("=" * 80)
        print(f"\n{pivot.to_string()}\n")

        if "No PCA" in pivot.columns:
            print("Improvement over 'No PCA' baseline:")
            print("-" * 80)
            baseline = pivot["No PCA"]
            for col in pivot.columns:
                if col == "No PCA":
                    continue
                improvement = pivot[col] - baseline
                print(f"\n{col}:")
                for model, delta in improvement.items():
                    sign = "+" if delta > 0 else ""
                    print(f"  {model:<25}: {sign}{delta:7.2f}{unit}")

        print("\n" + "=" * 80)

    def print_detailed_metrics_table(self):
        """Print comprehensive metrics table for all results."""
        print("\n" + "=" * 100)
        print("DETAILED METRICS — ALL VARIANTS & MODELS")
        print("=" * 100)

        for variant in sorted(self.results_df["Variant"].unique()):
            print(f"\n{variant}")
            print("-" * 100)

            variant_data = self.results_df[
                self.results_df["Variant"] == variant
            ].sort_values("Accuracy", ascending=False)

            header = f"{'Model':<25}" + "".join(f" | {m:>10}" for m in METRICS)
            print(header)
            print("-" * 100)
            for _, row in variant_data.iterrows():
                line = f"{row['Model']:<25}" + "".join(
                    f" | {row[m]:>10.4f}" for m in METRICS
                )
                print(line)

        print("\n" + "=" * 100)

    def print_summary_statistics(self):
        """Print mean/std/min/max for Accuracy, Precision, Recall, F1, grouped by variant."""
        print("\n" + "=" * 80)
        print("SUMMARY STATISTICS")
        print("=" * 80)

        summary_stats = (
            self.results_df.groupby("Variant")[
                ["Accuracy", "Precision", "Recall", "F1"]
            ]
            .agg(["mean", "std", "min", "max"])
            .round(4)
        )

        for metric in ["Accuracy", "Precision", "Recall", "F1"]:
            print(f"\n{metric} Statistics by Variant:")
            print(summary_stats[metric].to_string())

        print("\n" + "-" * 80)
        print("BEST MODEL BY METRIC")
        print("-" * 80)
        for metric in ["Accuracy", "F1", "Recall", "Precision"]:
            row = self.results_df.loc[self.results_df[metric].idxmax()]
            print(f"\n{metric}: {row['Model']} ({row['Variant']}) = {row[metric]:.4f}")

        print("\n" + "=" * 80)

    # ------------------------------------------------------------------
    # Plots
    # ------------------------------------------------------------------

    def plot_model_comparison_bars(self, metric: str = "Accuracy"):
        """
        Plot grouped bar chart comparing models across variants.

        Parameters
        ----------
        metric : str, default='Accuracy'
            Metric to plot: 'Accuracy', 'F1', 'Recall', 'Precision', etc.
        """
        pivot = self.results_df.pivot(index="Model", columns="Variant", values=metric)

        fig, ax = plt.subplots(figsize=(12, 6))
        x = np.arange(len(pivot.index))
        width = 0.25

        for idx, (variant, color) in enumerate(zip(pivot.columns, VARIANT_COLORS)):
            offset = (idx - 1) * width
            bars = ax.bar(
                x + offset, pivot[variant], width, label=variant, color=color, alpha=0.8
            )
            for bar in bars:
                height = bar.get_height()
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    height,
                    f"{height:.3f}",
                    ha="center",
                    va="bottom",
                    fontsize=9,
                )

        ax.set_xlabel("Model", fontsize=11, fontweight="bold")
        ax.set_ylabel(metric, fontsize=11, fontweight="bold")
        ax.set_title(
            f"{metric} Comparison Across Feature Spaces", fontsize=12, fontweight="bold"
        )
        ax.set_xticks(x)
        ax.set_xticklabels(pivot.index)
        ax.legend(loc="lower right", fontsize=10)
        ax.grid(axis="y", alpha=0.3)
        ax.set_ylim(0, 1.05)

        plt.tight_layout()
        plt.show()

    def plot_all_comparison_bars(
        self, metrics: List[str] = ("Accuracy", "F1", "Recall")
    ):
        """Convenience wrapper: plot_model_comparison_bars for several metrics in a row."""
        for metric in metrics:
            self.plot_model_comparison_bars(metric=metric)

    def plot_roc_comparison(self, y_test: np.ndarray, variant: str = "No PCA"):
        """
        Plot ROC curves for all models in a specific variant.

        Parameters
        ----------
        y_test : np.ndarray
            True test labels — passed explicitly rather than read from
            notebook global scope, so this method works regardless of
            what's currently bound to `y_test` outside the class.
        variant : str, default='No PCA'
            Variant to plot ROC curves for
        """
        variant_results = [r for r in self.binary_results if r["Variant"] == variant]
        if not variant_results:
            print(f"Warning: No results found for variant '{variant}'")
            return

        fig, ax = plt.subplots(figsize=(10, 8))

        for result in variant_results:
            y_proba = result.get("_y_test_proba")
            if y_proba is None:
                print(f"Warning: No probabilities for {result['Model']}")
                continue

            fpr, tpr, _ = roc_curve(y_test, y_proba)
            roc_auc = auc(fpr, tpr)
            model_name = result["Model"]
            ax.plot(
                fpr,
                tpr,
                color=MODEL_COLORS.get(model_name, "gray"),
                lw=2.5,
                label=f"{model_name} (AUC={roc_auc:.4f})",
            )

        ax.plot([0, 1], [0, 1], "k--", alpha=0.3, lw=1.5, label="Chance")
        ax.set_xlabel("False Positive Rate", fontsize=11, fontweight="bold")
        ax.set_ylabel("True Positive Rate", fontsize=11, fontweight="bold")
        ax.set_title(f"ROC Curves — {variant}", fontsize=12, fontweight="bold")
        ax.legend(loc="lower right", fontsize=10)
        ax.grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

    def plot_no_pca_feature_importance(
        self, no_pca_models: Dict[str, object], top_n: int = 15, figsize=(10, 7)
    ):
        """Plot feature importance/coefficients for all three No-PCA models in one call
        instead of three near-identical calls at the script level."""
        visualizer.plot_feature_importance(
            model=no_pca_models["Decision Tree"],
            feature_names=self.feature_names,
            title=f"Top {top_n} Features — Decision Tree (No PCA)",
            top_n=top_n,
            figsize=figsize,
        )
        visualizer.plot_feature_importance(
            model=no_pca_models["Random Forest"],
            feature_names=self.feature_names,
            title=f"Top {top_n} Features — Random Forest (No PCA)",
            top_n=top_n,
            figsize=figsize,
        )
        visualizer.plot_logreg_coefficients(
            model=no_pca_models["Logistic Regression"],
            feature_names=self.feature_names,
            title=f"Top {top_n} Coefficients — Logistic Regression (No PCA)",
            top_n=top_n,
            figsize=figsize,
        )

    def plot_no_pca_confusion_matrices(self, y_test: np.ndarray, figsize=(8, 7)):
        """Reconstruct predictions from stored probabilities/thresholds and plot a
        confusion matrix per model for the No-PCA variant."""
        no_pca_results_list = [
            r for r in self.binary_results if r["Variant"] == "No PCA"
        ]

        for model_name in MODEL_ORDER:
            model_result = next(
                (r for r in no_pca_results_list if r["Model"] == model_name), None
            )
            if model_result is None:
                print(f"Warning: No result found for {model_name}")
                continue

            y_proba = model_result.get("_y_test_proba")
            if y_proba is None:
                print(f"Warning: No probability predictions for {model_name}")
                continue

            threshold = model_result.get("Threshold", 0.5)
            y_pred = (y_proba >= threshold).astype(int)

            if len(y_pred) != len(y_test):
                print(
                    f"Error: Shape mismatch for {model_name}: {len(y_pred)} vs {len(y_test)}"
                )
                continue

            visualizer.plot_confusion_matrix(
                y_true=y_test,
                y_pred=y_pred,
                labels=["Normal", "Attack"],
                title=f"Confusion Matrix — {model_name} (No PCA)",
                figsize=figsize,
                save_name=f"cm_binary_{model_name.replace(' ', '_')}_no_pca",
            )
            print(f"✓ Confusion matrix for {model_name}")

        print("✓ All confusion matrices generated")

    # ------------------------------------------------------------------
    # Extraction helpers
    # ------------------------------------------------------------------

    def _extract_no_pca_field(self, field: str) -> Dict[str, object]:
        """Shared logic for pulling a per-model field (fitted model, confusion
        matrix, etc.) out of the No-PCA results. Used by both public extractors
        below, which previously duplicated this loop with only the field name
        and error message differing."""
        no_pca_results = [r for r in self.binary_results if r["Variant"] == "No PCA"]

        extracted = {}
        for model_name in MODEL_ORDER:
            value = next(
                (r[field] for r in no_pca_results if r["Model"] == model_name), None
            )
            if value is None:
                raise ValueError(
                    f"{field} for '{model_name}' not found in No-PCA results"
                )
            extracted[model_name] = value

        return extracted

    def extract_no_pca_models(self) -> Dict[str, object]:
        """
        Extract fitted models from No-PCA variant for interpretation.

        Returns
        -------
        dict
            Fitted models: {'Decision Tree': ..., 'Random Forest': ..., 'Logistic Regression': ...}
        """
        return self._extract_no_pca_field("_fitted_model")

    def extract_no_pca_confusion_matrices(self) -> Dict[str, np.ndarray]:
        """
        Extract confusion matrices from No-PCA variant.

        Returns
        -------
        dict
            Confusion matrices keyed by model name
        """
        return self._extract_no_pca_field("_confusion_matrix")

    def print_pca_equivalence_check(self, pca_comparison_results: Dict):
        """
        Print equivalence check between sklearn PCA and custom PCA.

        Parameters
        ----------
        pca_comparison_results : dict
            Results from PCAComparator.fit_and_compare()
        """
        equiv = pca_comparison_results["equivalence"]

        print("\n" + "=" * 80)
        print("PCA IMPLEMENTATION EQUIVALENCE CHECK")
        print("=" * 80)
        print(f"\nMax difference (transformed data): {equiv['max_diff_overall']:.2e}")
        print(f"Variance diff: {equiv['var_diff']:.2e}")
        print(f"Verdict: {equiv['verdict']}")
        print(
            "\n✓ Both PCA implementations produce mathematically equivalent results."
            "\n  Custom NumPy implementation is production-ready and mathematically correct."
        )
        print("\n" + "=" * 80)


# ============================================================
# Reporting helper (script section)
# ============================================================


def _print_pca_equivalence_summary(sklearn_pca, custom_pca, comparison_results):
    print("\n" + "=" * 80)
    print("PCA IMPLEMENTATION EQUIVALENCE")
    print("=" * 80)

    print(f"\nsklearn PCA:")
    print(f"  Components: {sklearn_pca.n_components_}")
    print(f"  Variance retained: {sklearn_pca.explained_variance_ratio_.sum():.4f}")
    print(f"  Fit+transform time: {comparison_results['sklearn']['runtime']:.4f}s")

    print(f"\nCustom PCA:")
    print(f"  Components: {custom_pca.n_components_}")
    print(f"  Variance retained: {custom_pca.explained_variance_ratio_.sum():.4f}")
    print(f"  Fit+transform time: {comparison_results['custom']['runtime']:.4f}s")

    equivalence = comparison_results["equivalence"]
    print(f"\nEquivalence Check:")
    print(f"  Max difference: {equivalence['max_diff_overall']:.2e}")
    print(f"  Verdict: {equivalence['verdict']}")
    print("\n" + "=" * 80)


# ============================================================
# Execute Analysis & Visualization
# ============================================================


def run_binary_results_analysis(
    binary_results, x_all, sklearn_pca, custom_pca, comparison_results, y_test
):
    """
    Run the full post-experiment analysis pipeline: tables, PCA equivalence
    summary, feature-importance plots, confusion matrices, comparison bars,
    ROC curves, and summary statistics.

    Returns
    -------
    dict with keys: analyzer, no_pca_models
    """
    print("\n" + "=" * 90)
    print("POST-EXPERIMENT ANALYSIS")
    print("=" * 90)

    analyzer = BinaryResultsAnalyzer(
        binary_results=binary_results, feature_names=list(x_all.columns)
    )

    analyzer.print_accuracy_comparison(
        include_variants=["No PCA", "PCA (sklearn)", "PCA (custom)"], as_percentage=True
    )
    analyzer.print_detailed_metrics_table()

    _print_pca_equivalence_summary(sklearn_pca, custom_pca, comparison_results)

    print("\n" + "=" * 80)
    print("FEATURE IMPORTANCE ANALYSIS (No-PCA Variant)")
    print("=" * 80)
    no_pca_models = analyzer.extract_no_pca_models()
    analyzer.plot_no_pca_feature_importance(no_pca_models)

    print("\n" + "=" * 80)
    print("CONFUSION MATRIX VISUALIZATIONS (No-PCA Variant)")
    print("=" * 80)
    analyzer.plot_no_pca_confusion_matrices(y_test)

    print("\n" + "=" * 80)
    print("GENERATING COMPARISON VISUALIZATIONS")
    print("=" * 80)
    analyzer.plot_all_comparison_bars(["Accuracy", "F1", "Recall"])
    print("✓ Comparison visualizations generated")

    print("\nGenerating ROC curves...")
    analyzer.plot_roc_comparison(y_test, variant="No PCA")

    analyzer.print_summary_statistics()

    print("\n" + "=" * 80)
    print("✓ Binary analysis complete")
    print("=" * 80 + "\n")

    binary_analysis_data = {"analyzer": analyzer, "no_pca_models": no_pca_models}

    print("Available for downstream cells:")
    for key, val in binary_analysis_data.items():
        if isinstance(val, dict):
            print(f"  • {key}: {list(val.keys())}")
        else:
            print(f"  • {key}: {type(val).__name__}")

    return binary_analysis_data


binary_analysis_data = run_binary_results_analysis(
    binary_results, x_all, sklearn_pca, custom_pca, comparison_results, y_test
)
analyzer = binary_analysis_data["analyzer"]
no_pca_models = binary_analysis_data["no_pca_models"]

In [ ]:
# ============================================================
# Multiclass Classification — Data Preparation
# ============================================================
# Extends binary classification pipeline to 5-category task:
# Normal, DoS, Probe, R2L, U2R
#
# Uses same 60/20/20 split strategy with stratification
# Reuses hyperparameters tuned on binary task (DT/RF)
# Maintains no-leakage discipline (scaler & PCA fit only on train)
# ============================================================

from typing import Dict, Tuple

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def _as_array(x):
    """Normalize DataFrame or ndarray input to a plain ndarray.
    Used at each call site that previously repeated
    `x.values if isinstance(x, pd.DataFrame) else x`."""
    return x.values if isinstance(x, pd.DataFrame) else x


class MulticlassDataPreparator:
    """
    Prepare data for multiclass classification with stratification.

    Manages:
    - Stratified train/test/val splits for multiclass labels
    - Feature scaling (separate from binary scaler)
    - PCA dimensionality reduction (separate from binary PCA)
    - Data validation and logging
    """

    def __init__(self, class_names, random_state: int = RANDOM_SEED):
        """
        Initialize preparator.

        Parameters
        ----------
        class_names : array-like
            Human-readable class names, indexed by encoded label
            (e.g. ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']). Stored here
            instead of read from notebook global scope, so printing works
            regardless of what `class_names` currently means outside this class.
        random_state : int, default=RANDOM_SEED
            Random state for reproducibility
        """
        self.class_names = class_names
        self.random_state = random_state
        self.scaler = None
        self.pca = None
        self.split_data = {}

    def split_data_stratified(
        self,
        x_data: pd.DataFrame,
        y_data: np.ndarray,
        train_size: float = 0.60,
        val_size: float = 0.20,
        test_size: float = 0.20,
        verbose: bool = True,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """
        Create stratified train/val/test splits for multiclass data.

        Parameters
        ----------
        x_data : pd.DataFrame
            Feature matrix
        y_data : np.ndarray
            Multiclass labels (0, 1, 2, 3, 4)
        train_size, val_size, test_size : float
            Split fractions; must sum to 1.0
        verbose : bool, default=True
            Print split information

        Returns
        -------
        tuple
            (x_train, x_val, x_test, y_train, y_val, y_test)
        """
        total_size = train_size + val_size + test_size
        assert np.isclose(total_size, 1.0), f"Sizes must sum to 1.0, got {total_size}"

        # Step 1: split test from train+val
        x_train_val, x_test, y_train_val, y_test = train_test_split(
            x_data,
            y_data,
            test_size=test_size,
            stratify=y_data,
            random_state=self.random_state,
        )

        # Step 2: split train from val — gives 0.80*0.75=0.60 train, 0.80*0.25=0.20 val
        val_fraction = val_size / (train_size + val_size)
        x_train, x_val, y_train, y_val = train_test_split(
            x_train_val,
            y_train_val,
            test_size=val_fraction,
            stratify=y_train_val,
            random_state=self.random_state,
        )

        self.split_data = {
            "x_train": x_train,
            "x_val": x_val,
            "x_test": x_test,
            "y_train": y_train,
            "y_val": y_val,
            "y_test": y_test,
        }

        if verbose:
            self._print_split_summary(y_train, y_val, y_test)

        return x_train, x_val, x_test, y_train, y_val, y_test

    def _print_split_summary(self, y_train, y_val, y_test):
        """Print detailed split statistics."""
        total = len(y_train) + len(y_val) + len(y_test)

        print("\n" + "=" * 80)
        print("MULTICLASS DATA SPLIT SUMMARY")
        print("=" * 80)

        print(f"\nOverall Split")
        print("-" * 80)
        print(
            f"  Train: {len(y_train):8,d} ({100 * len(y_train) / total:6.2f}%) | "
            f"Val: {len(y_val):8,d} ({100 * len(y_val) / total:6.2f}%) | "
            f"Test: {len(y_test):8,d} ({100 * len(y_test) / total:6.2f}%)"
        )

        print(f"\nClass Distribution by Split")
        print("-" * 80)
        for set_name, y_set in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
            print(f"\n{set_name}:")
            counts = np.bincount(y_set)
            for cls_idx, count in enumerate(counts):
                pct = 100 * count / len(y_set)
                bar = "█" * int(pct / 2)
                print(
                    f"  Class {cls_idx} ({self.class_names[cls_idx]:10s}): {count:6,d} ({pct:6.2f}%) {bar}"
                )

        print("\n" + "=" * 80)

    def scale_features(
        self,
        x_train: np.ndarray,
        x_val: np.ndarray,
        x_test: np.ndarray,
        verbose: bool = True,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Scale features using StandardScaler (fit on train only).

        Parameters
        ----------
        x_train, x_val, x_test : np.ndarray or pd.DataFrame
            Unscaled feature arrays
        verbose : bool, default=True
            Print scaling information

        Returns
        -------
        tuple
            (x_train_scaled, x_val_scaled, x_test_scaled)
        """
        x_train, x_val, x_test = _as_array(x_train), _as_array(x_val), _as_array(x_test)

        self.scaler = StandardScaler()
        x_train_scaled = self.scaler.fit_transform(x_train.astype(np.float64))
        x_val_scaled = self.scaler.transform(x_val.astype(np.float64))
        x_test_scaled = self.scaler.transform(x_test.astype(np.float64))

        if verbose:
            print("\n" + "=" * 80)
            print("FEATURE SCALING (Multiclass Task)")
            print("=" * 80)
            print(f"\n✓ Scaler fit ONLY on training data (no data leakage)")
            print(
                f"  Training samples mean: {np.mean(x_train_scaled, axis=0).mean():.6f}"
            )
            print(
                f"  Training samples std:  {np.std(x_train_scaled, axis=0).mean():.6f}"
            )
            print(f"  Val set successfully transformed")
            print(f"  Test set successfully transformed")
            print("\n" + "=" * 80)

        return x_train_scaled, x_val_scaled, x_test_scaled

    def apply_pca(
        self,
        x_train_scaled: np.ndarray,
        x_val_scaled: np.ndarray,
        x_test_scaled: np.ndarray,
        n_components: str = "mle",
        verbose: bool = True,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Apply PCA to scaled features (fit on train only).

        Parameters
        ----------
        x_train_scaled, x_val_scaled, x_test_scaled : np.ndarray
            Scaled feature arrays
        n_components : str or int, default='mle'
            Number of components or 'mle'
        verbose : bool, default=True
            Print PCA information

        Returns
        -------
        tuple
            (x_train_pca, x_val_pca, x_test_pca)
        """
        self.pca = SklearnPCA(
            n_components=n_components, svd_solver="full", random_state=self.random_state
        )

        x_train_pca = self.pca.fit_transform(x_train_scaled)
        x_val_pca = self.pca.transform(x_val_scaled)
        x_test_pca = self.pca.transform(x_test_scaled)

        if verbose:
            explained = self.pca.explained_variance_ratio_.sum()
            print("\n" + "=" * 80)
            print("PRINCIPAL COMPONENT ANALYSIS (Multiclass Task)")
            print("=" * 80)
            print(f"\nPCA Configuration")
            print(f"  N-components specified: {n_components}")
            print(f"  Components selected: {self.pca.n_components_}")
            print(f"  Variance retained: {explained:.4f} ({explained * 100:.2f}%)")
            print(f"\nPCA Transformed Shapes")
            print(f"  Train: {x_train_pca.shape}")
            print(f"  Val:   {x_val_pca.shape}")
            print(f"  Test:  {x_test_pca.shape}")
            print(f"\n✓ PCA fit ONLY on training data (no data leakage)")
            print("=" * 80)

        return x_train_pca, x_val_pca, x_test_pca

    def get_split_data(self) -> Dict:
        """Get stored split data."""
        return self.split_data.copy()


# ============================================================
# Validation & Summary (script section)
# ============================================================


def _print_array_integrity(name: str, arr: np.ndarray) -> None:
    """One line of NaN/Inf counts for a named array. Replaces three
    near-identical print calls (train/val/test scaled arrays)."""
    print(f"  {name}: NaN={np.isnan(arr).sum()}, Inf={np.isinf(arr).sum()}")


def _print_validation_report(
    x_tr_mc_scaled,
    x_val_mc_scaled,
    x_test_mc_scaled,
    x_tr_mc_pca,
    x_val_mc_pca,
    x_test_mc_pca,
    y_tr_mc,
    y_val_mc,
    y_test_mc,
):
    print("\n" + "=" * 90)
    print("DATA PREPARATION VALIDATION")
    print("=" * 90)

    print(f"\nData Integrity Checks:")
    _print_array_integrity("x_tr_mc_scaled", x_tr_mc_scaled)
    _print_array_integrity("x_val_mc_scaled", x_val_mc_scaled)
    _print_array_integrity("x_test_mc_scaled", x_test_mc_scaled)

    print(f"\nLabel Integrity Checks:")
    for name, y in [
        ("y_tr_mc", y_tr_mc),
        ("y_val_mc", y_val_mc),
        ("y_test_mc", y_test_mc),
    ]:
        print(f"  {name} min/max: {y.min()}/{y.max()}")

    print(f"\nFinal Shapes (No-PCA):")
    print(f"  x_tr_mc_scaled:  {x_tr_mc_scaled.shape}")
    print(f"  x_val_mc_scaled: {x_val_mc_scaled.shape}")
    print(f"  x_test_mc_scaled: {x_test_mc_scaled.shape}")

    print(f"\nFinal Shapes (With PCA):")
    print(f"  x_tr_mc_pca:  {x_tr_mc_pca.shape}")
    print(f"  x_val_mc_pca: {x_val_mc_pca.shape}")
    print(f"  x_test_mc_pca: {x_test_mc_pca.shape}")

    train_mean = np.mean(x_tr_mc_scaled, axis=0)
    val_mean = np.mean(x_val_mc_scaled, axis=0)
    test_mean = np.mean(x_test_mc_scaled, axis=0)

    print(f"\nData Leakage Check (Scaler fitted on train only):")
    print(f"  Train mean close to 0: {np.allclose(train_mean, 0, atol=1e-10)}")
    print(f"  Val mean close to 0:   {np.allclose(val_mean, 0, atol=0.1)}")
    print(f"  Test mean close to 0:  {np.allclose(test_mean, 0, atol=0.1)}")
    print(f"  ✓ No data leakage (validation/test NOT used for scaling)")

    print("\n" + "=" * 90)
    print("✓ Multiclass data preparation complete")
    print("=" * 90 + "\n")


# ============================================================
# Execute Multiclass Data Preparation
# ============================================================


def prepare_multiclass_data(
    x_all, y_all_multiclass, class_names, random_state=RANDOM_SEED
):
    """
    Run the full multiclass prep pipeline: stratified split -> scale -> PCA,
    then validate and report. Returns everything downstream cells need.
    """
    print("\n" + "=" * 90)
    print("MULTICLASS CLASSIFICATION — DATA PREPARATION")
    print("=" * 90)

    mc_preparator = MulticlassDataPreparator(
        class_names=class_names, random_state=random_state
    )

    x_tr_mc, x_val_mc, x_test_mc, y_tr_mc, y_val_mc, y_test_mc = (
        mc_preparator.split_data_stratified(
            x_data=x_all,
            y_data=y_all_multiclass,
            train_size=0.60,
            val_size=0.20,
            test_size=0.20,
            verbose=True,
        )
    )

    x_tr_mc_scaled, x_val_mc_scaled, x_test_mc_scaled = mc_preparator.scale_features(
        x_train=x_tr_mc, x_val=x_val_mc, x_test=x_test_mc, verbose=True
    )

    x_tr_mc_pca, x_val_mc_pca, x_test_mc_pca = mc_preparator.apply_pca(
        x_train_scaled=x_tr_mc_scaled,
        x_val_scaled=x_val_mc_scaled,
        x_test_scaled=x_test_mc_scaled,
        n_components="mle",
        verbose=True,
    )

    _print_validation_report(
        x_tr_mc_scaled,
        x_val_mc_scaled,
        x_test_mc_scaled,
        x_tr_mc_pca,
        x_val_mc_pca,
        x_test_mc_pca,
        y_tr_mc,
        y_val_mc,
        y_test_mc,
    )

    multiclass_data = {
        "preparator": mc_preparator,
        "x_tr_mc_scaled": x_tr_mc_scaled,
        "x_val_mc_scaled": x_val_mc_scaled,
        "x_test_mc_scaled": x_test_mc_scaled,
        "x_tr_mc_pca": x_tr_mc_pca,
        "x_val_mc_pca": x_val_mc_pca,
        "x_test_mc_pca": x_test_mc_pca,
        "y_tr_mc": y_tr_mc,
        "y_val_mc": y_val_mc,
        "y_test_mc": y_test_mc,
        "pca_mc": mc_preparator.pca,
        "scaler_mc": mc_preparator.scaler,
    }

    print("Available for downstream cells:")
    for key, val in multiclass_data.items():
        if isinstance(val, (np.ndarray, pd.DataFrame)):
            print(f"  • {key}: {val.shape}")
        else:
            print(f"  • {key}: {type(val).__name__}")

    return multiclass_data


multiclass_data = prepare_multiclass_data(
    x_all, y_all_multiclass, class_names, random_state=RANDOM_SEED
)

mc_preparator = multiclass_data["preparator"]
x_tr_mc_scaled = multiclass_data["x_tr_mc_scaled"]
x_val_mc_scaled = multiclass_data["x_val_mc_scaled"]
x_test_mc_scaled = multiclass_data["x_test_mc_scaled"]
x_tr_mc_pca = multiclass_data["x_tr_mc_pca"]
x_val_mc_pca = multiclass_data["x_val_mc_pca"]
x_test_mc_pca = multiclass_data["x_test_mc_pca"]
y_tr_mc = multiclass_data["y_tr_mc"]
y_val_mc = multiclass_data["y_val_mc"]
y_test_mc = multiclass_data["y_test_mc"]
pca_mc = multiclass_data["pca_mc"]
scaler_mc = multiclass_data["scaler_mc"]

In [ ]:
# ============================================================
# Multiclass Classification Experiments — Feature Space Variants
# ============================================================
# Compare model performance across:
#   1. No PCA (original 41 features)
#   2. PCA (~40 components, 100% variance)
#
# Reuses hyperparameters tuned on binary task (DT/RF/LR)
# Evaluates on 5-category target (Normal, DoS, Probe, R2L, U2R)
# ============================================================

from datetime import datetime
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

METRICS = ["Accuracy", "Macro F1", "Weighted F1", "Macro Precision", "Macro Recall"]
MODEL_ORDER = ["Decision Tree", "Random Forest", "Logistic Regression"]
MINORITY_CLASSES = ["U2R", "R2L"]  # low-support classes worth alerting on


class MulticlassExperimentRunner:
    """
    Orchestrate multiclass classification experiments across feature spaces.

    Manages:
    - Multiple feature space variants
    - Model instantiation with tuned hyperparameters
    - Result collection and comparison
    - Per-class performance analysis
    - Experiment tracking
    """

    def __init__(self, class_names: np.ndarray, random_state: int = RANDOM_SEED):
        """
        Initialize experiment runner.

        Parameters
        ----------
        class_names : np.ndarray
            Array of class names (e.g., ['Normal', 'DoS', 'Probe', 'R2L', 'U2R'])
        random_state : int, default=RANDOM_SEED
            Random state for reproducibility
        """
        self.class_names = class_names
        self.random_state = random_state
        self.results = []
        self.experiment_log = []

    def run_all_variants(
        self,
        feature_variants: Dict[str, Tuple],
        y_train: np.ndarray,
        y_test: np.ndarray,
        best_params: Dict,
        verbose: bool = True,
    ) -> List[Dict]:
        """
        Run multiclass classification across all feature space variants.

        Parameters
        ----------
        feature_variants : dict
            Mapping of variant names to (x_train, x_test) tuples
        y_train, y_test : np.ndarray
            Multiclass labels (0-4) for each set
        best_params : dict
            Best hyperparameters for each model
        verbose : bool, default=True
            Print detailed results for each variant

        Returns
        -------
        list
            List of result dictionaries from all experiments
        """
        if verbose:
            print("\n" + "=" * 90)
            print("MULTICLASS CLASSIFICATION EXPERIMENTS")
            print("=" * 90)
            print(f"\nConfiguration:")
            print(f"  • Feature space variants: {len(feature_variants)}")
            print(f"  • Training samples: {len(y_train):,}")
            print(f"  • Test samples: {len(y_test):,}")
            print(f"  • Number of classes: {len(self.class_names)}")
            print(f"  • Class names: {', '.join(self.class_names)}")
            print(f"  • Class distribution (train):")
            for cls_idx, cls_name in enumerate(self.class_names):
                count = (y_train == cls_idx).sum()
                pct = 100 * count / len(y_train)
                print(f"      {cls_name:10s}: {count:6,d} ({pct:6.2f}%)")
            print()

        self.results = []

        for variant_idx, (variant_label, feature_data) in enumerate(
            feature_variants.items(), 1
        ):
            x_train_v, x_test_v = feature_data

            if verbose:
                print(f"\n[{variant_idx}/{len(feature_variants)}] {variant_label}")
                print(f"  Feature shape: {x_train_v.shape}")

            models = build_models(
                dt_params=best_params["Decision Tree"],
                rf_params=best_params["Random Forest"],
                lr_params=best_params["Logistic Regression"],
                random_state=self.random_state,
            )

            variant_results = run_multiclass_experiment(
                models=models,
                x_train=x_train_v,
                y_train=y_train,
                x_test=x_test_v,
                y_test=y_test,
                class_names=self.class_names,
                variant_label=variant_label,
            )
            self.results.extend(variant_results)

            self.experiment_log.append(
                {
                    "variant": variant_label,
                    "n_features": x_train_v.shape[1],
                    "n_models": len(models),
                    "completed_at": datetime.now(),
                }
            )

        if verbose:
            print("\n" + "=" * 90)

        return self.results

    def create_comparison_dataframe(self) -> pd.DataFrame:
        """
        Create comparison DataFrame from all experiments.

        Returns
        -------
        pd.DataFrame
            Results with columns: Model, Variant, Accuracy, Macro F1, Weighted F1, ...
        """
        return pd.DataFrame(self.results)

    def _results_or_warn(self) -> pd.DataFrame | None:
        """Shared guard: return the results dataframe, or None + a message if empty.
        Replaces the identical `if not self.results: print(...); return` check
        that was repeated at the top of six methods below."""
        if not self.results:
            print("No results available")
            return None
        return self.create_comparison_dataframe()

    def print_accuracy_comparison(self):
        """Print overall accuracy comparison across models and variants."""
        results_df = self._results_or_warn()
        if results_df is None:
            return

        print("\n" + "=" * 90)
        print("ACCURACY COMPARISON (Multiclass)")
        print("=" * 90)

        accuracy_pivot_pct = (
            results_df.pivot(index="Model", columns="Variant", values="Accuracy") * 100
        ).round(2)

        print("\nAccuracy (%):\n")
        print(accuracy_pivot_pct.to_string())

        if "No PCA" in accuracy_pivot_pct.columns:
            print("\nImprovement over 'No PCA' baseline:")
            baseline = accuracy_pivot_pct["No PCA"]
            for col in accuracy_pivot_pct.columns:
                if col == "No PCA":
                    continue
                improvement = accuracy_pivot_pct[col] - baseline
                print(f"\n{col}:")
                for model, delta in improvement.items():
                    sign = "+" if delta > 0 else ""
                    print(f"  {model:<25}: {sign}{delta:6.2f}%")

        print("\n" + "=" * 90)

    def print_detailed_results(self):
        """Print comprehensive results with macro and weighted metrics."""
        results_df = self._results_or_warn()
        if results_df is None:
            return

        print("\n" + "=" * 90)
        print("DETAILED MULTICLASS RESULTS")
        print("=" * 90)

        for variant in sorted(results_df["Variant"].unique()):
            print(f"\n{variant}")
            print("-" * 90)

            variant_data = results_df[results_df["Variant"] == variant].sort_values(
                "Accuracy", ascending=False
            )

            header = f"{'Model':<25}" + "".join(f" | {m:>12}" for m in METRICS)
            print(header)
            print("-" * 90)
            for _, row in variant_data.iterrows():
                line = f"{row['Model']:<25}" + "".join(
                    f" | {row[m]:>12.4f}" for m in METRICS
                )
                print(line)

        print("\n" + "=" * 90)

    def print_per_class_analysis(self):
        """
        Print per-class precision, recall, F1 for all models and variants.

        Critical for identifying minority-class (U2R, R2L) weaknesses.
        """
        results_df = self._results_or_warn()
        if results_df is None:
            return

        print("\n" + "=" * 90)
        print("PER-CLASS PERFORMANCE ANALYSIS")
        print("=" * 90)

        for variant in sorted(results_df["Variant"].unique()):
            print(f"\n{variant}")
            print("-" * 90)

            variant_results = [r for r in self.results if r["Variant"] == variant]

            for result in variant_results:
                print(f"\n{result['Model']}")
                report = result["_per_class_report"]

                print(
                    f"  {'Class':<15} | {'Precision':>10} | {'Recall':>10} | {'F1-Score':>10} | {'Support':>10}"
                )
                print(
                    f"  {'-' * 15}-+-{'-' * 10}-+-{'-' * 10}-+-{'-' * 10}-+-{'-' * 10}"
                )

                for cls_name in self.class_names:
                    cls_report = report[cls_name]
                    print(
                        f"  {cls_name:<15} | {cls_report['precision']:>10.4f} | {cls_report['recall']:>10.4f} | "
                        f"{cls_report['f1-score']:>10.4f} | {int(cls_report['support']):>10d}"
                    )

        print("\n" + "=" * 90)

    def print_best_models(self, metric: str = "Weighted F1"):
        """
        Identify and print best models by metric.

        Parameters
        ----------
        metric : str, default='Weighted F1'
            Metric to rank by (important for imbalanced multiclass)
        """
        results_df = self._results_or_warn()
        if results_df is None:
            return

        print(f"\n" + "=" * 90)
        print(f"BEST MODELS BY {metric}")
        print("=" * 90)

        top_models = results_df.nlargest(5, metric)[
            ["Model", "Variant", metric, "Accuracy", "Macro F1"]
        ]

        for rank, (_, row) in enumerate(top_models.iterrows(), 1):
            print(
                f"\n{rank}. {row['Model']} ({row['Variant']})"
                f"\n   {metric}: {row[metric]:.4f}, Accuracy: {row['Accuracy']:.4f}, "
                f"Macro F1: {row['Macro F1']:.4f}"
            )

        print("\n" + "=" * 90)

    def print_minority_class_alert(self, threshold_recall: float = 0.50):
        """
        Alert on poor minority-class performance.

        Parameters
        ----------
        threshold_recall : float, default=0.50
            Alert if recall below this threshold
        """
        if not self.results:
            print("No results available")
            return

        print("\n" + "=" * 90)
        print("MINORITY CLASS ALERT SYSTEM")
        print("=" * 90)

        alerts = []
        for result in self.results:
            report = result.get("_per_class_report", {})
            for cls_name in MINORITY_CLASSES:
                if cls_name not in report:
                    continue
                recall = report[cls_name]["recall"]
                if recall < threshold_recall:
                    alerts.append(
                        {
                            "Model": result["Model"],
                            "Variant": result["Variant"],
                            "Class": cls_name,
                            "Recall": recall,
                            "Support": int(report[cls_name]["support"]),
                        }
                    )

        if alerts:
            print(
                f"\n⚠️  ALERTS: {len(alerts)} minority-class models below {threshold_recall * 100:.0f}% recall\n"
            )
            alerts_df = pd.DataFrame(alerts).sort_values("Recall")
            print(alerts_df.to_string(index=False))
        else:
            print(
                f"\n✓ No alerts: All minority classes above {threshold_recall * 100:.0f}% recall"
            )

        print("\n" + "=" * 90)

    def get_variant_data(self, variant_name: str) -> pd.DataFrame:
        """Get results for a specific variant."""
        results_df = self.create_comparison_dataframe()
        return results_df[results_df["Variant"] == variant_name]

    def compare_variants_for_model(self, model_name: str):
        """Print comparison of all variants for a specific model."""
        results_df = self._results_or_warn()
        if results_df is None:
            return

        model_data = results_df[results_df["Model"] == model_name]

        print(f"\n" + "=" * 90)
        print(f"{model_name} — VARIANT COMPARISON (Multiclass)")
        print("=" * 90)

        print(f"{'Variant':<25} | " + " | ".join(f"{m:>12}" for m in METRICS))
        print("-" * 90)
        for _, row in model_data.iterrows():
            print(
                f"{row['Variant']:<25} | "
                + " | ".join(f"{row[m]:>12.4f}" for m in METRICS)
            )

        print("\n" + "=" * 90)


# ============================================================
# Reporting helpers (script section)
# ============================================================


def _print_group_stats(
    results_df: pd.DataFrame, group_col: str, value_col: str, label: str
) -> None:
    """Print mean/std/min/max of value_col grouped by group_col. Replaces
    three near-identical stat-printing loops (variant-accuracy, model-accuracy,
    variant-weighted-F1) that previously differed only in these two columns."""
    print(f"\n\n{label}:")
    for group_value in sorted(results_df[group_col].unique()):
        subset = results_df[results_df[group_col] == group_value][value_col]
        print(
            f"\n{group_value}:"
            f"\n  Mean:   {subset.mean():.4f}"
            f"\n  Std:    {subset.std():.4f}"
            f"\n  Min:    {subset.min():.4f}"
            f"\n  Max:    {subset.max():.4f}"
        )


def _print_statistical_summary(results_df: pd.DataFrame) -> None:
    print("\n" + "=" * 90)
    print("STATISTICAL SUMMARY (Multiclass)")
    print("=" * 90)

    _print_group_stats(
        results_df, "Variant", "Accuracy", "Accuracy statistics by variant"
    )
    _print_group_stats(results_df, "Model", "Accuracy", "Accuracy statistics by model")
    _print_group_stats(
        results_df, "Variant", "Weighted F1", "Weighted F1-Score statistics by variant"
    )

    print("\n" + "=" * 90)
    print("✓ Multiclass classification experiments complete")
    print("=" * 90 + "\n")


# ============================================================
# Execute Multiclass Experiments
# ============================================================


def run_multiclass_experiments_pipeline(
    x_tr_mc_scaled,
    x_test_mc_scaled,
    x_tr_mc_pca,
    x_test_mc_pca,
    y_tr_mc,
    y_test_mc,
    class_names,
    best_params,
):
    """Run the full multiclass comparison across feature-space variants and
    print all standard reports. Returns everything downstream cells need."""
    print("\n" + "=" * 90)
    print("PREPARING MULTICLASS CLASSIFICATION EXPERIMENTS")
    print("=" * 90)

    multiclass_variants = {
        "No PCA": (x_tr_mc_scaled, x_test_mc_scaled),
        "PCA": (x_tr_mc_pca, x_test_mc_pca),
    }

    print(f"\nFeature space variants:")
    for variant_name, (x_tr_v, x_te_v) in multiclass_variants.items():
        print(f"  • {variant_name:<15}: Train {x_tr_v.shape}, Test {x_te_v.shape}")

    mc_runner = MulticlassExperimentRunner(
        class_names=class_names, random_state=RANDOM_SEED
    )
    multiclass_results = mc_runner.run_all_variants(
        feature_variants=multiclass_variants,
        y_train=y_tr_mc,
        y_test=y_test_mc,
        best_params=best_params,
        verbose=True,
    )

    mc_runner.print_accuracy_comparison()
    mc_runner.print_detailed_results()
    mc_runner.print_per_class_analysis()
    mc_runner.print_best_models(metric="Weighted F1")
    mc_runner.print_minority_class_alert(threshold_recall=0.50)

    print("\n" + "=" * 90)
    print("MODEL-SPECIFIC VARIANT ANALYSIS")
    print("=" * 90)
    for model_name in MODEL_ORDER:
        mc_runner.compare_variants_for_model(model_name)

    results_df = mc_runner.create_comparison_dataframe()
    _print_statistical_summary(results_df)

    multiclass_experiment_data = {
        "runner": mc_runner,
        "multiclass_results": multiclass_results,
        "results_df": results_df,
        "multiclass_variants": multiclass_variants,
    }

    print("Available for downstream cells:")
    for key, val in multiclass_experiment_data.items():
        if isinstance(val, pd.DataFrame):
            print(f"  • {key}: {val.shape}")
        elif isinstance(val, list):
            print(f"  • {key}: {len(val)} items")
        else:
            print(f"  • {key}: {type(val).__name__}")

    return multiclass_experiment_data


multiclass_experiment_data = run_multiclass_experiments_pipeline(
    x_tr_mc_scaled,
    x_test_mc_scaled,
    x_tr_mc_pca,
    x_test_mc_pca,
    y_tr_mc,
    y_test_mc,
    class_names,
    BEST_PARAMS,
)

mc_runner = multiclass_experiment_data["runner"]
multiclass_results = multiclass_experiment_data["multiclass_results"]
results_df = multiclass_experiment_data["results_df"]
multiclass_variants = multiclass_experiment_data["multiclass_variants"]

In [ ]:
# ============================================================
# Final Multiclass Comparison & Analysis (Refactored)
# ============================================================

import pandas as pd
from sklearn.metrics import confusion_matrix

SEP_WIDE = "=" * 90
SEP_THIN = "-" * 90

METRIC_COLS = [
    "Accuracy",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "Weighted Precision",
    "Weighted Recall",
    "Weighted F1",
]

DEFAULT_VARIANT_ORDER = ("No PCA", "PCA")
DEFAULT_MINORITY_CLASSES = ("U2R", "R2L")


# ------------------------------------------------------------
# Small shared helpers
# ------------------------------------------------------------


def print_header(title, char="="):
    """Print a title wrapped in a full-width separator line."""
    sep = char * 90
    print(f"\n{sep}\n{title}\n{sep}")


def build_metric_pivot(mc_df, metric, variant_order=DEFAULT_VARIANT_ORDER):
    """Pivot a single metric into a Model x Variant table.

    Returns None (and prints a warning) if the pivot fails, so callers
    can just check for None instead of wrapping every call in try/except.
    """
    try:
        pivot = mc_df.pivot(index="Model", columns="Variant", values=metric)
    except Exception as e:
        print(f"Error pivoting {metric}: {e}")
        return None

    available = [v for v in variant_order if v in pivot.columns]
    return pivot[available]


# ------------------------------------------------------------
# Section 11a: All metrics comparison
# ------------------------------------------------------------


def print_metric_improvement(pivot, baseline_col="No PCA"):
    """Print how each non-baseline variant differs from the baseline, per model."""
    if baseline_col not in pivot.columns or pivot.shape[1] < 2:
        return

    baseline = pivot[baseline_col]
    print("  Improvement over 'No PCA':")
    for col in pivot.columns:
        if col == baseline_col:
            continue
        for model, delta in (pivot[col] - baseline).items():
            sign = "+" if delta > 0 else ""
            print(f"    {model} → {col}: {sign}{delta:.4f}")


def print_all_metrics_comparison(mc_df, metric_cols=METRIC_COLS):
    print_header("SECTION 11a: ALL METRICS COMPARISON")

    for metric in metric_cols:
        pivot = build_metric_pivot(mc_df, metric)
        if pivot is None:
            continue

        is_accuracy = metric == "Accuracy"
        display = (pivot * 100).round(2) if is_accuracy else pivot.round(4)
        label = f"{metric} (%)" if is_accuracy else metric

        print(f"\n{label}:")
        print(display.to_string())
        print_metric_improvement(pivot)


# ------------------------------------------------------------
# Section 11b: Per-class recall analysis
# ------------------------------------------------------------


def build_per_class_df(multiclass_results, class_names):
    """Flatten each model's per-class report into one long-format DataFrame."""
    rows = [
        {
            "Model": r["Model"],
            "Variant": r["Variant"],
            "Class": cls,
            "Recall": r["_per_class_report"][cls]["recall"],
            "Precision": r["_per_class_report"][cls]["precision"],
            "F1": r["_per_class_report"][cls]["f1-score"],
            "Support": int(r["_per_class_report"][cls]["support"]),
        }
        for r in multiclass_results
        for cls in class_names
    ]
    return pd.DataFrame(rows)


def print_recall_pivot(per_class_df):
    print("\nRECALL by Model, Class, and Variant:")
    print(SEP_THIN)
    try:
        pivot = per_class_df.pivot_table(
            index=["Model", "Class"], columns="Variant", values="Recall"
        )
        print(pivot.round(4).to_string())
    except Exception as e:
        print(f"Error creating recall pivot: {e}")


def print_support_table(per_class_df, preferred_variant="No PCA"):
    print("\n\nCLASS SUPPORT (Test Set Distribution):")
    print(SEP_THIN)
    pivot = per_class_df.pivot_table(
        index=["Model", "Class"], columns="Variant", values="Support"
    )
    column = (
        preferred_variant if preferred_variant in pivot.columns else pivot.columns[0]
    )
    print(pivot[column].to_string())


def print_minority_class_alert(per_class_df, minority_classes, recall_threshold=0.50):
    print_header("MINORITY CLASS ALERT (U2R & R2L)")

    poor_performers = per_class_df[
        per_class_df["Class"].isin(minority_classes)
        & (per_class_df["Recall"] < recall_threshold)
    ]

    if poor_performers.empty:
        print(
            f"\n✓ All models achieve ≥{recall_threshold:.0%} recall on minority classes"
        )
        return

    print(
        f"\n⚠️  WARNING: {len(poor_performers)} model(s) with "
        f"<{recall_threshold:.0%} recall on minority classes:\n"
    )
    print(
        poor_performers[["Model", "Variant", "Class", "Recall", "Support"]].to_string(
            index=False
        )
    )


def print_best_minority_performance(per_class_df, minority_classes):
    print("\n\nBEST MINORITY CLASS PERFORMANCE:")
    print(SEP_THIN)

    for cls_name in minority_classes:
        cls_data = per_class_df[per_class_df["Class"] == cls_name].sort_values(
            "Recall", ascending=False
        )
        if cls_data.empty:
            continue

        best = cls_data.iloc[0]
        print(f"\n{cls_name}:")
        print(
            f"  Best: {best['Model']} ({best['Variant']}) - Recall: {best['Recall']:.4f}, "
            f"Precision: {best['Precision']:.4f}, Support: {int(best['Support'])}"
        )


def print_per_class_recall_analysis(
    multiclass_results, class_names, minority_classes=DEFAULT_MINORITY_CLASSES
):
    print_header(
        "SECTION 11b: PER-CLASS RECALL ANALYSIS (Critical for Minority Classes)"
    )

    per_class_df = build_per_class_df(multiclass_results, class_names)
    print_recall_pivot(per_class_df)
    print_support_table(per_class_df)
    print_minority_class_alert(per_class_df, minority_classes)
    print_best_minority_performance(per_class_df, minority_classes)


# ------------------------------------------------------------
# Section 11c: Confusion matrices
# ------------------------------------------------------------


def plot_no_pca_confusion_matrices(
    multiclass_results, class_names, x_test_mc_scaled, y_test_mc, visualizer
):
    print_header("SECTION 11c: CONFUSION MATRICES (No-PCA Variant, All Models)")

    no_pca_results = [r for r in multiclass_results if r["Variant"] == "No PCA"]
    if not no_pca_results:
        print("\nWarning: No 'No PCA' variant results found")
        return

    for result in no_pca_results:
        try:
            model = result["_fitted_model"]
            y_pred = model.predict(x_test_mc_scaled)

            visualizer.plot_confusion_matrix(
                y_true=y_test_mc,
                y_pred=y_pred,
                labels=list(class_names),
                title=f"Confusion Matrix — {result['Model']} (Multiclass, No PCA)",
                figsize=(10, 9),
                save_name=f"cm_multiclass_{result['Model'].replace(' ', '_')}_no_pca",
            )
        except Exception as e:
            print(f"Error plotting confusion matrix for {result['Model']}: {e}")


# ------------------------------------------------------------
# Static text sections (insights & summary)
# ------------------------------------------------------------


def print_key_insights():
    print_header("KEY INSIGHTS & RECOMMENDATIONS")
    print("""
1. METRIC INTERPRETATION
   • Accuracy: Overall correctness (dominated by majority classes)
   • Macro-F1: Equal weight to all classes (reveals minority issues)
   • Weighted-F1: Weighted by class frequency (best for imbalanced data)

2. MINORITY CLASS DETECTION (Critical)
   • U2R: Only 24 samples in test set — expect high variance in metrics
   • R2L: 776 samples — still challenging but more stable
   • ALWAYS check per-class recall for U2R/R2L before deployment
   • High overall accuracy can mask poor minority class detection

3. PCA EFFECT ON MULTICLASS
   • Compare both variants systematically
   • Watch for PCA hurting minority class recall (common issue)
   • PCA may help with majority classes but hurt minority detection

4. MODEL SELECTION FOR PRODUCTION
   • Don't rely solely on accuracy or weighted-F1
   • Ensure acceptable recall on minority classes (typically ≥50%)
   • Consider macro-F1 for balanced performance
   • Test-time latency matters: Decision Trees < Random Forest < Logistic Regression

5. DATA QUALITY ASSURANCE
   ✓ No data leakage:
     - Scaler fit only on training data
     - PCA fit only on training data
     - Hyperparameters from binary task (no re-tuning on multiclass)
   ✓ Stratified splits preserve class distribution
   ✓ Validation set unused (binary class tuning only)
   ✓ All randomness seeded for reproducibility
""")


def print_summary(binary_results, multiclass_results):
    print_header("SUMMARY")
    print(f"""
✓ Experiments completed:
  • Binary classification: {len(binary_results)} model-variant pairs
  • Multiclass classification: {len(multiclass_results)} model-variant pairs
  • Total models trained & evaluated: {len(binary_results) + len(multiclass_results)}

✓ Data integrity verified:
  • No training/test data leakage
  • Stratified cross-validation throughout
  • Consistent random seed for reproducibility

✓ Comprehensive metrics:
  • Binary: Accuracy, Precision, Recall, F1, Sensitivity, Specificity
  • Multiclass: Accuracy, Macro/Weighted P/R/F1, Per-class breakdown
  • Minority class focus: U2R (24 samples), R2L (776 samples)

✓ Ready for:
  • Model deployment (with minority class monitoring)
  • Further hyperparameter tuning if needed
  • Ensemble methods or hybrid approaches
  • Production monitoring and retraining pipeline
""")
    print(SEP_WIDE)
    print("✓ Analysis complete. Ready for conclusion and deployment recommendations.")
    print(SEP_WIDE + "\n")


# ------------------------------------------------------------
# Orchestrator
# ------------------------------------------------------------


def run_multiclass_analysis(
    multiclass_results,
    binary_results,
    class_names,
    x_test_mc_scaled,
    y_test_mc,
    visualizer,
    minority_classes=DEFAULT_MINORITY_CLASSES,
):
    """Run the full multiclass comparison & analysis report, section by section."""
    print_header("FINAL MULTICLASS COMPARISON TABLE")
    mc_df = pd.DataFrame(multiclass_results)

    print_all_metrics_comparison(mc_df)
    print_per_class_recall_analysis(multiclass_results, class_names, minority_classes)
    plot_no_pca_confusion_matrices(
        multiclass_results, class_names, x_test_mc_scaled, y_test_mc, visualizer
    )
    print_key_insights()
    print_summary(binary_results, multiclass_results)


# ------------------------------------------------------------
# Entry point — replace this call with your actual variables
# ------------------------------------------------------------

run_multiclass_analysis(
    multiclass_results=multiclass_results,
    binary_results=binary_results,
    class_names=class_names,
    x_test_mc_scaled=x_test_mc_scaled,
    y_test_mc=y_test_mc,
    visualizer=visualizer,
)

In [ ]:
from sklearn.base import clone


def fit_and_predict(model, x_train, y_train, x_test):
    """
    Fit a fresh clone of the model on training data and return
    predictions on test data. Cloning avoids leaking state between
    repeated calls (e.g. in a bootstrap loop).

    Parameters
    ----------
    model : sklearn estimator
        Unfitted model instance (used as a template; not mutated)
    x_train : np.ndarray
        Training features
    y_train : np.ndarray
        Training labels
    x_test : np.ndarray
        Test features

    Returns
    -------
    np.ndarray
        Predictions on test data
    """
    fitted_model = clone(model).fit(x_train, y_train)
    return fitted_model.predict(x_test)

In [ ]:
# ============================================================
# Bootstrap Confidence Intervals — Minority Class Recall
# ============================================================
# Computes 95% CI for U2R & R2L recall across all model-variant pairs
# Uses stratified bootstrap resampling (resamples within each class)
# Provides uncertainty quantification for small minority classes
#
# NOTE ON SCOPE: this resampling scheme only draws from samples where
# y_true == target_class, so it never sees false positives. That makes
# it statistically valid for RECALL only — a previous version also
# exposed 'precision'/'f1' options, but their bootstrap estimates were
# invalid (false positives were structurally impossible to sample), so
# that support has been removed here rather than silently kept broken.
# ============================================================

import warnings
from typing import Dict, Optional

import numpy as np
import pandas as pd
from sklearn.metrics import recall_score

MIN_WARN_SAMPLES = 10
MIN_RECOMMENDED_SAMPLES = 30


class BootstrapCICalculator:
    """
    Bootstrap confidence intervals for per-class recall.

    Uses stratified (within-class) resampling: for each target class,
    resamples are drawn only from that class's true-label samples, then
    recall is recomputed on each resample.
    """

    def __init__(
        self,
        n_bootstrap: int = 2000,
        ci_percentile: float = 95.0,
        random_state: int = RANDOM_SEED,
    ):
        """
        Parameters
        ----------
        n_bootstrap : int, default=2000
            Number of bootstrap resamples (1000-10000 typical)
        ci_percentile : float, default=95.0
            Confidence level (e.g., 95.0 for 95% CI)
        random_state : int, default=RANDOM_SEED
            Random seed for reproducibility
        """
        self.n_bootstrap = n_bootstrap
        self.ci_percentile = ci_percentile
        self.random_state = random_state
        self.results = []

    def compute_bootstrap_ci(
        self, y_true: np.ndarray, y_pred: np.ndarray, target_class: int
    ) -> Optional[Dict]:
        """
        Compute bootstrap CI for one class's recall.

        Returns
        -------
        dict or None
            None if the class has no samples in y_true.
        """
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        class_indices = np.nonzero(y_true == target_class)[0]
        n_class = len(class_indices)
        if n_class == 0:
            return None

        if n_class < MIN_WARN_SAMPLES:
            warnings.warn(
                f"Class {target_class} has only {n_class} samples — "
                f"bootstrap CI may be unreliable (recommend n >= {MIN_RECOMMENDED_SAMPLES})"
            )

        point_estimate = recall_score(
            y_true, y_pred, labels=[target_class], average="micro", zero_division=0
        )

        # Vectorized bootstrap: draw all n_bootstrap resamples in one shot
        # instead of looping in Python (much faster for large n_bootstrap).
        rng = np.random.default_rng(self.random_state)
        resampled_idx = rng.choice(
            class_indices, size=(self.n_bootstrap, n_class), replace=True
        )
        bootstrap_recalls = np.mean(y_pred[resampled_idx] == target_class, axis=1)

        alpha = 100 - self.ci_percentile
        ci_lower, ci_upper = np.percentile(
            bootstrap_recalls, [alpha / 2, 100 - alpha / 2]
        )

        return {
            "n_samples": n_class,
            "point_estimate": point_estimate,
            "bootstrap_mean": bootstrap_recalls.mean(),
            "bootstrap_std": bootstrap_recalls.std(),
            "bootstrap_median": np.median(bootstrap_recalls),
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
            "ci_width": ci_upper - ci_lower,
            "ci_percentile": self.ci_percentile,
        }

    def compute_all_models(
        self,
        models_predictions: Dict,
        y_true: np.ndarray,
        class_indices: Dict[str, int],
        verbose: bool = True,
    ) -> pd.DataFrame:
        """
        Compute bootstrap recall CIs for all model-variant-class combinations.

        Parameters
        ----------
        models_predictions : dict
            Mapping of (model_name, variant) -> predictions
        y_true : np.ndarray
            True labels
        class_indices : dict
            Mapping of class_name -> class index
        verbose : bool, default=True
            Print progress

        Returns
        -------
        pd.DataFrame
        """
        y_true = np.asarray(y_true)
        results = []
        total = len(models_predictions) * len(class_indices)
        count = 0

        if verbose:
            _print_header(
                f"BOOTSTRAP CONFIDENCE INTERVALS ({self.n_bootstrap} resamples)"
            )

        for (model_name, variant), y_pred in models_predictions.items():
            for cls_name, cls_idx in class_indices.items():
                count += 1
                if verbose:
                    print(
                        f"\n[{count}/{total}] {model_name:20s} — {variant:10s} — {cls_name:10s}",
                        end=" ... ",
                    )

                ci_result = self.compute_bootstrap_ci(y_true, y_pred, cls_idx)
                if ci_result is None:
                    if verbose:
                        print("SKIPPED (no samples)")
                    continue

                results.append(
                    {
                        "Model": model_name,
                        "Variant": variant,
                        "Class": cls_name,
                        "N_Samples": ci_result["n_samples"],
                        "Point_Estimate": ci_result["point_estimate"],
                        "Bootstrap_Mean": ci_result["bootstrap_mean"],
                        "Bootstrap_Std": ci_result["bootstrap_std"],
                        "Bootstrap_Median": ci_result["bootstrap_median"],
                        "CI_Lower": ci_result["ci_lower"],
                        "CI_Upper": ci_result["ci_upper"],
                        "CI_Width": ci_result["ci_width"],
                    }
                )

                if verbose:
                    print(
                        f"Point: {ci_result['point_estimate']:.4f}, "
                        f"95% CI: [{ci_result['ci_lower']:.4f}, {ci_result['ci_upper']:.4f}]"
                    )

        if verbose:
            print("\n" + "=" * 90)

        self.results = results
        return pd.DataFrame(results)

    def print_results_table(self, df: pd.DataFrame, variant_filter: str = None):
        """Print a formatted CI results table, optionally filtered to one variant."""
        if variant_filter:
            df_display = df[df["Variant"] == variant_filter]
            title = f"Bootstrap CI Results — {variant_filter}"
        else:
            df_display = df
            title = "Bootstrap CI Results — All Variants"

        _print_header(title, width=120)

        numeric_cols = [
            "Point_Estimate",
            "Bootstrap_Mean",
            "Bootstrap_Std",
            "CI_Lower",
            "CI_Upper",
            "CI_Width",
        ]
        df_fmt = df_display.copy()
        # .applymap was removed in newer pandas (2.1+); .map is the replacement
        df_fmt[numeric_cols] = df_fmt[numeric_cols].map(lambda x: f"{x:.4f}")

        print(df_fmt.to_string(index=False))
        print("=" * 120)

    def test_statistical_significance(
        self, df: pd.DataFrame, verbose: bool = True
    ) -> pd.DataFrame:
        """Compare CI overlap between the two variants for each model/class pair."""
        if verbose:
            _print_header("STATISTICAL SIGNIFICANCE TEST — CI Overlap")

        significance_results = [
            self._compare_variants(model, cls, group, verbose)
            for model in df["Model"].unique()
            for cls in df["Class"].unique()
            if (group := df[(df["Model"] == model) & (df["Class"] == cls)])[
                "Variant"
            ].nunique()
            == 2
        ]

        if verbose:
            print("\n" + "=" * 90)

        return pd.DataFrame(significance_results)

    @staticmethod
    def _compare_variants(
        model: str, cls: str, group: pd.DataFrame, verbose: bool
    ) -> Dict:
        """Compare the two variants within one model/class group; report CI overlap."""
        variant1, variant2 = sorted(group["Variant"].unique())
        row1 = group[group["Variant"] == variant1].iloc[0]
        row2 = group[group["Variant"] == variant2].iloc[0]

        overlap = not (
            row1["CI_Upper"] < row2["CI_Lower"] or row2["CI_Upper"] < row1["CI_Lower"]
        )
        effect_size = row1["Point_Estimate"] - row2["Point_Estimate"]

        if verbose:
            verdict = (
                "NO OVERLAP ✗ (significant)"
                if not overlap
                else "OVERLAP ✓ (not significant)"
            )
            print(f"\n{model} — {cls}:")
            print(
                f"  {variant1:15s}: {row1['Point_Estimate']:.4f} [{row1['CI_Lower']:.4f}, {row1['CI_Upper']:.4f}]"
            )
            print(
                f"  {variant2:15s}: {row2['Point_Estimate']:.4f} [{row2['CI_Lower']:.4f}, {row2['CI_Upper']:.4f}]"
            )
            print(f"  Effect size: {effect_size:+.4f}")
            print(f"  Verdict: {verdict}")

        return {
            "Model": model,
            "Class": cls,
            "Variant_1": variant1,
            "Variant_2": variant2,
            "Point_Estimate_1": row1["Point_Estimate"],
            "Point_Estimate_2": row2["Point_Estimate"],
            "Effect_Size": effect_size,
            "CI_Overlap": overlap,
            "Significant": not overlap,
        }

    def print_interpretation_guide(self):
        """Print guidance on interpreting bootstrap CI results."""
        _print_header("BOOTSTRAP CI INTERPRETATION GUIDE")
        print("""
POINT ESTIMATE vs CI WIDTH:
  • Point Estimate: Single best guess from test predictions
  • CI Width: Uncertainty due to small test set size
  • Narrow CI: High confidence in point estimate
  • Wide CI: High uncertainty (common for small classes like U2R)

INTERPRETING FOR MINORITY CLASSES:
  • U2R (24 samples): Expect VERY WIDE CI — point estimate unreliable alone
  • R2L (776 samples): Expect wider CI than majority classes but more stable
  • Always report CI alongside point estimate for transparency

STATISTICAL SIGNIFICANCE:
  • Non-overlapping CIs: Statistically significant difference (p < 0.05 approx)
  • Overlapping CIs: No significant difference detected
  • Note: Absence of significance ≠ absence of effect (may need larger sample)

PRACTICAL IMPLICATIONS:
  1. Don't trust single point estimate for small classes
  2. Report CI as [lower, upper] range for honesty
  3. Use effect size (difference in point estimates) alongside CI overlap
  4. Consider practical significance (2% improvement may not be worth it)
  5. For production, ensure minority class recall meets business requirements

BOOTSTRAP METHOD NOTES:
  • Distribution-free, works for any sample size
  • Pros: no normality assumption, handles small samples
  • Cons: needs enough samples (n ≥ 10 recommended, n ≥ 30 ideal)
  • Percentile method used here (simple, intuitive)
""")
        print("=" * 90)


# ------------------------------------------------------------
# Shared print helper
# ------------------------------------------------------------


def _print_header(title, width=90, char="="):
    sep = char * width
    print(f"\n{sep}\n{title}\n{sep}")


# ============================================================
# Execute Bootstrap Analysis
# ============================================================

_print_header("BOOTSTRAP CONFIDENCE INTERVALS FOR MINORITY CLASS RECALL")
print("\nFitting models for bootstrap CI (one per model-variant pair)...")

# Model configs: (name, sklearn class, constructor kwargs). Add a model by
# adding one tuple here instead of duplicating a whole fit_and_predict call.
MODEL_CONFIGS = [
    (
        "Decision Tree",
        DecisionTreeClassifier,
        dict(
            max_depth=dt_params["max_depth"],
            min_samples_split=dt_params["min_samples_split"],
            min_samples_leaf=dt_params["min_samples_leaf"],
            criterion=dt_params["criterion"],
            ccp_alpha=0.0,
            random_state=RANDOM_SEED,
        ),
    ),
    (
        "Random Forest",
        RandomForestClassifier,
        dict(
            n_estimators=rf_params["n_estimators"],
            max_depth=rf_params["max_depth"],
            min_samples_leaf=rf_params["min_samples_leaf"],
            max_features=rf_params["max_features"],
            random_state=RANDOM_SEED,
            n_jobs=-1,
        ),
    ),
    (
        "Logistic Regression",
        LogisticRegression,
        dict(
            C=lr_params["C"],
            solver="lbfgs",
            max_iter=lr_params["max_iter"],
            random_state=RANDOM_SEED,
        ),
    ),
]

# Variant name -> (x_train, x_test) for that variant
VARIANT_DATA = {
    "No PCA": (x_tr_mc_scaled, x_test_mc_scaled),
    "PCA": (x_tr_mc_pca, x_test_mc_pca),
}

models_predictions = {
    (model_name, variant): fit_and_predict(
        model_cls(**params), x_train, y_tr_mc, x_test
    )
    for model_name, model_cls, params in MODEL_CONFIGS
    for variant, (x_train, x_test) in VARIANT_DATA.items()
}

print("✓ Models fit complete")

# Initialize calculator
calculator = BootstrapCICalculator(n_bootstrap=2000, random_state=RANDOM_SEED)

# Minority classes only (U2R and R2L)
minority_class_indices = {
    "U2R": list(class_names).index("U2R"),
    "R2L": list(class_names).index("R2L"),
}

ci_results_df = calculator.compute_all_models(
    models_predictions=models_predictions,
    y_true=y_test_mc,
    class_indices=minority_class_indices,
    verbose=True,
)

calculator.print_results_table(ci_results_df)
significance_df = calculator.test_statistical_significance(ci_results_df, verbose=True)
calculator.print_interpretation_guide()

# ============================================================
# Summary Statistics
# ============================================================

_print_header("SUMMARY STATISTICS")

print("\nMinority Class Support:")
for cls_name, cls_idx in minority_class_indices.items():
    count = (y_test_mc == cls_idx).sum()
    pct = 100 * count / len(y_test_mc)
    print(f"  {cls_name}: {count:4d} samples ({pct:5.2f}%)")


def print_ci_width_analysis(ci_results_df, minority_classes=("U2R", "R2L")):
    print("\nCI Width Analysis (smaller is better):")
    for cls_name in minority_classes:
        cls_data = ci_results_df[ci_results_df["Class"] == cls_name]
        if cls_data.empty:
            print(f"\n  {cls_name}: No data available")
            continue

        min_row = cls_data.loc[cls_data["CI_Width"].idxmin()]
        max_row = cls_data.loc[cls_data["CI_Width"].idxmax()]

        print(f"\n  {cls_name}:")
        print(f"    Mean CI width: {cls_data['CI_Width'].mean():.4f}")
        print(f"    Min CI width:  {min_row['CI_Width']:.4f} ({min_row['Model']})")
        print(f"    Max CI width:  {max_row['CI_Width']:.4f} ({max_row['Model']})")


def print_best_minority_recall(ci_results_df, minority_classes=("U2R", "R2L")):
    print("\n\nBest Minority Class Recall (by point estimate):")
    for cls_name in minority_classes:
        cls_data = ci_results_df[ci_results_df["Class"] == cls_name]
        if cls_data.empty:
            print(f"\n  {cls_name}: No data available")
            continue

        best = cls_data.loc[cls_data["Point_Estimate"].idxmax()]
        print(f"\n  {cls_name}:")
        print(
            f"    {best['Model']:20s} ({best['Variant']:10s}): {best['Point_Estimate']:.4f} "
            f"[{best['CI_Lower']:.4f}, {best['CI_Upper']:.4f}]"
        )


print_ci_width_analysis(ci_results_df)
print_best_minority_recall(ci_results_df)

_print_header("✓ Bootstrap CI analysis complete")

# Store results for downstream cells
bootstrap_ci_data = {
    "calculator": calculator,
    "ci_results_df": ci_results_df,
    "significance_df": significance_df,
}

print("Available for downstream cells:")
for key, val in bootstrap_ci_data.items():
    label = f"{val.shape}" if isinstance(val, pd.DataFrame) else type(val).__name__
    print(f"  • {key}: {label}")

In [ ]:
# ============================================================
# VISUALIZATION SUITE — 7 diagnostic plots for the NSL-KDD study
# Uses variables already computed in earlier blocks:
#   sk_pca, x_tr_scaled, x_test_scaled, x_test_pca_sklearn,
#   binary_results, multiclass_results, results_df (bootstrap CI),
#   dt_no_pca_model, rf_no_pca_model, class_names, y_test, y_test_mc
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Seaborn's default categorical palette — index into it by position
# instead of hardcoding hex values, so colors stay consistent even
# if more models/variants are added later.
PALETTE = sns.color_palette("deep")

COLOR_DT = PALETTE[0]  # Decision Tree
COLOR_RF = PALETTE[3]  # Random Forest
COLOR_NOPCA = PALETTE[2]  # No PCA variant
COLOR_PCA = PALETTE[1]  # PCA variant

In [ ]:
# 2D PCA for visualization
pca_2d = SklearnPCA(n_components=2, svd_solver="full", random_state=RANDOM_SEED)
x_tr_mc_2d = pca_2d.fit_transform(x_tr_mc_scaled)

# Single source of truth for per-class plot styling — order, size (marker area),
# and alpha (opacity) all live together, so adding/removing a class only
# requires editing one place instead of keeping three lists in sync.
CLASS_PLOT_STYLE = {
    "DoS": {"size": 5, "alpha": 0.15},
    "Normal": {"size": 5, "alpha": 0.15},
    "Probe": {"size": 8, "alpha": 0.30},
    "R2L": {"size": 15, "alpha": 0.60},
    "U2R": {"size": 40, "alpha": 0.90},
}

class_plot_order = list(CLASS_PLOT_STYLE.keys())
sizes = {cls: style["size"] for cls, style in CLASS_PLOT_STYLE.items()}
alphas = {cls: style["alpha"] for cls, style in CLASS_PLOT_STYLE.items()}

In [ ]:
# ============================================================
# BLOCK V0 — Before vs After PCA scatter (replaces your existing V0 cell)
# ============================================================
# %%
def scatter_by_class(ax, df, x_col, y_col, class_order, point_sizes, point_alphas):
    """Scatter one subplot's points, grouped by class, using per-class size/alpha."""
    for cls in class_order:
        subset = df[df["Class"] == cls]
        ax.scatter(
            subset[x_col],
            subset[y_col],
            label=cls,
            s=point_sizes[cls],
            alpha=point_alphas[cls],
            rasterized=True,  # keeps PDF/SVG file size sane with thousands of points
        )


class_labels = pd.Series(y_tr_mc).map(dict(enumerate(class_names))).values

plot_df_before = pd.DataFrame(
    {
        "Feature 1": x_tr_mc_scaled[:, 0],
        "Feature 2": x_tr_mc_scaled[:, 1],
        "Class": class_labels,
    }
)

plot_df_after = pd.DataFrame(
    {
        "PC1": x_tr_mc_2d[:, 0],
        "PC2": x_tr_mc_2d[:, 1],
        "Class": class_labels,
    }
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

scatter_by_class(
    axes[0], plot_df_before, "Feature 1", "Feature 2", class_plot_order, sizes, alphas
)
axes[0].set_title("Before PCA")
axes[0].set_xlabel("Standardized Feature 1")
axes[0].set_ylabel("Standardized Feature 2")

scatter_by_class(axes[1], plot_df_after, "PC1", "PC2", class_plot_order, sizes, alphas)
axes[1].set_title("After PCA")
axes[1].set_xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]*100:.2f}% variance)")
axes[1].set_ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]*100:.2f}% variance)")

handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(class_names))

plt.suptitle("Before vs After PCA Transformation", fontsize=14)
plt.tight_layout()
save_fig(fig, "v0_before_after_pca")
plt.show()


# ============================================================
# BLOCK V1 — Scree plot (replaces your existing V1 cell)
# ============================================================
# %%
def plot_scree(x_scaled, mle_n_components, color_bar, color_line):
    """Plot individual (bar) + cumulative (line) explained variance,
    with a marker at the MLE-selected component count."""
    pca_full = SklearnPCA(svd_solver="full", random_state=RANDOM_SEED)
    pca_full.fit(x_scaled)

    individual_var = pca_full.explained_variance_ratio_
    cumulative_var = np.cumsum(individual_var)
    n_components_range = np.arange(1, len(individual_var) + 1)

    fig, ax1 = plt.subplots(figsize=(10, 6))

    ax1.bar(
        n_components_range,
        individual_var,
        alpha=0.5,
        color=color_bar,
        label="Individual Explained Variance",
    )
    ax1.set_xlabel("Principal Component")
    ax1.set_ylabel("Individual Explained Variance Ratio", color=color_bar)
    ax1.tick_params(axis="y", labelcolor=color_bar)

    ax2 = ax1.twinx()
    ax2.plot(
        n_components_range,
        cumulative_var,
        color=color_line,
        marker="o",
        markersize=3,
        linewidth=2,
        label="Cumulative Explained Variance",
    )
    ax2.axvline(
        mle_n_components,
        color="gray",
        linestyle="--",
        alpha=0.7,
        label=f"MLE-selected n={mle_n_components}",
    )
    ax2.axhline(
        cumulative_var[mle_n_components - 1], color="gray", linestyle=":", alpha=0.5
    )
    ax2.set_ylabel("Cumulative Explained Variance Ratio", color=color_line)
    ax2.tick_params(axis="y", labelcolor=color_line)
    ax2.set_ylim(0, 1.05)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")

    plt.title("PCA Scree Plot — Individual & Cumulative Explained Variance")
    plt.tight_layout()
    save_fig(fig, "v1_scree_plot")
    plt.show()

    print(
        f"MLE selected {mle_n_components} components, capturing "
        f"{cumulative_var[mle_n_components - 1]*100:.2f}% of total variance."
    )


plot_scree(
    x_tr_scaled,
    mle_n_components=sklearn_pca.n_components_,
    color_bar=COLOR_DT,
    color_line=COLOR_RF,
)


# ============================================================
# BLOCK V2 — ROC curves, binary, 3 variants (replaces your existing V2 cell)
# ============================================================
# %%
from sklearn.metrics import roc_curve as roc_curve_fn, auc

variant_test_data = {
    "No PCA": x_test_scaled,
    "PCA (sklearn)": x_test_pca_sklearn,
    "PCA (custom)": x_test_pca_custom,
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharey=True)

for ax, (variant_label, x_te_v) in zip(axes, variant_test_data.items()):
    variant_results = [r for r in binary_results if r["Variant"] == variant_label]
    for r in variant_results:
        model = r["_fitted_model"]
        y_proba = model.predict_proba(x_te_v)[:, 1]
        fpr, tpr, _ = roc_curve_fn(y_test, y_proba)
        roc_auc = auc(fpr, tpr)
        ax.plot(
            fpr,
            tpr,
            linewidth=2,
            color=MODEL_COLORS.get(r["Model"], "gray"),
            label=f"{r['Model']} (AUC={roc_auc:.4f})",
        )

    ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Chance")
    ax.set_xlabel("False Positive Rate")
    ax.set_title(variant_label)
    ax.legend(loc="lower right", fontsize=9)

axes[0].set_ylabel("True Positive Rate")
plt.suptitle(
    "ROC Curves — Binary Classification (Normal vs Attack)", y=1.02, fontsize=13
)
plt.tight_layout()
save_fig(fig, "v2_roc_curves")
plt.show()


# ============================================================
# BLOCK V3a — PCA sensitivity sweep, compute (unchanged from your original —
# no plot here, kept for completeness so V3b has sweep_df available)
# ============================================================
# %%
variance_candidates = [0.999, 0.99, 0.98, 0.95, 0.90, 0.85]

pca_full_mc = SklearnPCA(svd_solver="full", random_state=RANDOM_SEED)
pca_full_mc.fit(x_tr_mc_scaled)
cum_var_mc = np.cumsum(pca_full_mc.explained_variance_ratio_)


def per_class_recall(model, x_te_, y_te_):
    pred = model.predict(x_te_)
    report = classification_report(
        y_te_, pred, target_names=class_names, output_dict=True, zero_division=0
    )
    return (
        report["U2R"]["recall"],
        report["R2L"]["recall"],
        report["macro avg"]["f1-score"],
    )


PREFIXES = ["DT", "RF", "LR"]


def fit_variant_and_get_recalls(x_train, x_test, y_train, y_test):
    models = build_models(dt_params, rf_params, lr_params, random_state=RANDOM_SEED)
    row = {}
    for prefix, model in zip(PREFIXES, models.values()):
        model.fit(x_train, y_train)
        u2r, r2l, macro_f1 = per_class_recall(model, x_test, y_test)
        row[f"{prefix}_U2R"] = u2r
        row[f"{prefix}_R2L"] = r2l
        row[f"{prefix}_MacroF1"] = macro_f1
    return row


sweep_rows = [
    {
        "Variance": 1.0,
        "n_components": x_tr_mc_scaled.shape[1],
        **fit_variant_and_get_recalls(
            x_tr_mc_scaled, x_test_mc_scaled, y_tr_mc, y_test_mc
        ),
    }
]

for var_target in variance_candidates:
    n_comp = int(np.nonzero(cum_var_mc >= var_target)[0][0] + 1)
    pca_v = SklearnPCA(n_components=n_comp, svd_solver="full", random_state=RANDOM_SEED)
    x_tr_v = pca_v.fit_transform(x_tr_mc_scaled)
    x_te_v = pca_v.transform(x_test_mc_scaled)
    sweep_rows.append(
        {
            "Variance": var_target,
            "n_components": n_comp,
            **fit_variant_and_get_recalls(x_tr_v, x_te_v, y_tr_mc, y_test_mc),
        }
    )

sweep_df = pd.DataFrame(sweep_rows).sort_values("n_components")
print(sweep_df.to_string(index=False))


# ============================================================
# BLOCK V3b — PCA sensitivity sweep, plot (replaces your existing V3b cell)
# ============================================================
# %%
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics_to_plot = [
    ("U2R", "U2R Recall"),
    ("R2L", "R2L Recall"),
    ("MacroF1", "Macro-F1"),
]

MODEL_LINE_STYLE = [
    ("DT", "Decision Tree", COLOR_DT, "o"),
    ("RF", "Random Forest", COLOR_RF, "s"),
    ("LR", "Logistic Regression", "purple", "^"),
]

for ax, (metric_key, metric_label) in zip(axes, metrics_to_plot):
    for prefix, label, color, marker in MODEL_LINE_STYLE:
        ax.plot(
            sweep_df["n_components"],
            sweep_df[f"{prefix}_{metric_key}"],
            marker=marker,
            color=color,
            label=label,
            linewidth=2,
        )
    ax.set_xlabel("Number of PCA Components")
    ax.set_ylabel(metric_label)
    ax.set_title(f"{metric_label} vs PCA Components")
    ax.legend()
    ax.invert_xaxis()

plt.suptitle("PCA Sensitivity Sweep — Multiclass Task", y=1.03, fontsize=13)
plt.tight_layout()
save_fig(fig, "v3_pca_sensitivity_sweep")
plt.show()


# ============================================================
# BLOCK V4 — Bootstrap CI error-bar chart (replaces your existing V4 cell)
# ============================================================
# %%
def get_ci_arrays(subset, models_order, variant):
    """Return (points, lower_err, upper_err) for one variant, ordered by models_order."""
    variant_data = subset[subset["Variant"] == variant].set_index("Model")
    missing = [m for m in models_order if m not in variant_data.index]
    if missing:
        raise ValueError(f"No bootstrap CI for variant='{variant}', models={missing}")

    row = variant_data.loc[models_order]
    points = row["Point_Estimate"].to_numpy()
    lower_err = points - row["CI_Lower"].to_numpy()
    upper_err = row["CI_Upper"].to_numpy() - points
    return points, lower_err, upper_err


fig, axes = plt.subplots(1, 2, figsize=(14, 6))

models_order = ["Decision Tree", "Random Forest", "Logistic Regression"]
variants_order = ["No PCA", "PCA"]
x_base = np.arange(len(models_order))
width = 0.3

for ax, cls_name in zip(axes, ["U2R", "R2L"]):
    subset = ci_results_df[ci_results_df["Class"] == cls_name]

    for i, variant in enumerate(variants_order):
        color = COLOR_NOPCA if variant == "No PCA" else COLOR_PCA
        offset = (i - 0.5) * width
        points, lower_err, upper_err = get_ci_arrays(subset, models_order, variant)

        ax.errorbar(
            x_base + offset,
            points,
            yerr=[lower_err, upper_err],
            fmt="o",
            markersize=8,
            capsize=6,
            capthick=2,
            linewidth=2,
            color=color,
            label=variant,
        )

    ax.set_xticks(x_base)
    ax.set_xticklabels(models_order)
    ax.set_ylabel("Recall")
    n_samples_this_class = subset["N_Samples"].iloc[0]
    ax.set_title(f"{cls_name} Recall — 95% Bootstrap CI (n={n_samples_this_class})")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Bootstrap 95% Confidence Intervals — No PCA vs PCA", y=1.03, fontsize=13)
plt.tight_layout()
save_fig(fig, "v4_bootstrap_ci")
plt.show()

u2r_n = ci_results_df.loc[ci_results_df["Class"] == "U2R", "N_Samples"].iloc[0]
r2l_n = ci_results_df.loc[ci_results_df["Class"] == "R2L", "N_Samples"].iloc[0]
print(f"Visual confirmation: wide, overlapping error bars for U2R (n={u2r_n}) mean the")
print("No-PCA vs PCA point-estimate difference is statistical noise. Narrow,")
print(f"non-overlapping error bars for R2L (n={r2l_n}) confirm a genuine PCA effect.")


# ============================================================
# BLOCK V5 — Class distribution log-scale bar chart (replaces your existing V5 cell)
# ============================================================
# %%
class_counts_train = pd.Series(y_tr_mc).map(dict(enumerate(class_names))).value_counts()
class_counts_test = (
    pd.Series(y_test_mc).map(dict(enumerate(class_names))).value_counts()
)

dist_df = (
    pd.DataFrame({"Train": class_counts_train, "Test": class_counts_test})
    .reindex(class_names)
    .fillna(0)
    .astype(int)
)

fig, ax = plt.subplots(figsize=(10, 6))
x_pos = np.arange(len(class_names))
width = 0.35

bars1 = ax.bar(
    x_pos - width / 2, dist_df["Train"], width, label="Train", color=COLOR_DT
)
bars2 = ax.bar(x_pos + width / 2, dist_df["Test"], width, label="Test", color=COLOR_RF)

ax.set_yscale("log")
ax.set_xticks(x_pos)
ax.set_xticklabels(class_names)
ax.set_ylabel("Sample Count (log scale)")
ax.set_title("Class Distribution — Train vs Test (Log Scale)")
ax.legend()

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.annotate(
                f"{int(height)}",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=8,
            )

plt.tight_layout()
save_fig(fig, "v5_class_distribution")
plt.show()

print(dist_df.to_string())

imbalance_ratio = dist_df["Train"].max() / dist_df["Train"].min()
print(f"\nMajority:Minority class imbalance ratio (Train): {imbalance_ratio:.1f}:1")

for cls_name in DEFAULT_MINORITY_CLASSES:
    support = dist_df.loc[cls_name, "Test"]
    pct = 100 * support / dist_df["Test"].sum()
    print(f"{cls_name} test support: {support} ({pct:.3f}% of test set)")


# ============================================================
# BLOCK V6 — Hero summary grouped bar chart (replaces your existing V6 cell)
# ============================================================
# %%
def annotate_bars(ax, *bar_containers):
    for bars in bar_containers:
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax.annotate(
                    f"{height:.3f}",
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha="center",
                    va="bottom",
                    fontsize=8,
                )


summary_rows = [
    {
        "Model": r["Model"],
        "Variant": r["Variant"],
        "Accuracy": r["Accuracy"],
        "Macro F1": r["Macro F1"],
        **{
            f"{cls} Recall": r["_per_class_report"][cls]["recall"]
            for cls in DEFAULT_MINORITY_CLASSES
        },
    }
    for r in multiclass_results
]
summary_df = pd.DataFrame(summary_rows)

metrics = ["Accuracy", "Macro F1"] + [
    f"{cls} Recall" for cls in DEFAULT_MINORITY_CLASSES
]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

for ax, metric in zip(axes.flatten(), metrics):
    pivot = summary_df.pivot(index="Model", columns="Variant", values=metric)
    pivot = pivot[["No PCA", "PCA"]]

    x_pos = np.arange(len(pivot.index))
    width = 0.35

    bars1 = ax.bar(
        x_pos - width / 2, pivot["No PCA"], width, label="No PCA", color=COLOR_NOPCA
    )
    bars2 = ax.bar(x_pos + width / 2, pivot["PCA"], width, label="PCA", color=COLOR_PCA)
    annotate_bars(ax, bars1, bars2)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(pivot.index)
    ax.set_title(metric)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle(
    "No PCA vs PCA — Summary Across Key Metrics (Multiclass Task)", y=1.00, fontsize=14
)
plt.tight_layout()
save_fig(fig, "v6_summary_grouped")
plt.show()